In [1]:
import numpy as np
import h5py
import pandas as pd

from tqdm import tqdm
import gzip
import re
import tempfile

#from lhereader import LHEReader

from LHCO_reader import LHCO_reader

import math
import os

# Plotting library
import matplotlib.pyplot as plt

# AUXILIARY FUNCTIONS

## PARTICLE ID CUTS

In [2]:
def basic_id_cuts(event):
    """
    Applies basic identification cuts
    
    Input:
     - a single LHCO reader event
    
    Returns:
     - the index of the particles that satisfy the basic identification conditions:
     photons, electrons, muons, taus, jets, btagged jets
    """

    photons   = event["photon"]
    electrons = event["electron"]
    muons     = event["muon"]
    taus      = event["tau"]
    jets      = event["jet"]

    # ---- PHOTONS ----
    index_photon = [
        ll for ll in range(len(photons))
        if photons[ll]["PT"] > 20 and ( -2.37 < photons[ll]["eta"] < -1.52 or
                                        -1.37 < photons[ll]["eta"] <  1.37 or
                                         1.52 < photons[ll]["eta"] <  2.37    )    ]

    # ---- ELECTRONS ----
    index_e = [
        ll for ll in range(len(electrons))
        if electrons[ll]["PT"] > 10 and -2.47 < electrons[ll]["eta"] < 2.47    ]

    # ---- MUONS ----
    index_mu = [
        ll for ll in range(len(muons))
        if muons[ll]["PT"] > 10 and -2.7 < muons[ll]["eta"] < 2.7    ]

    # ---- TAUS ----
    index_tau = [
        ll for ll in range(len(taus))
        if taus[ll]["PT"] > 20 and ( -2.5  < taus[ll]["eta"] < -1.52 or
                                     -1.37 < taus[ll]["eta"] <  1.37 or
                                      1.52 < taus[ll]["eta"] <  2.5     )    ]

    # ---- JETS ----
    index_jet = [
        ll for ll in range(len(jets))
        if jets[ll]["PT"] > 20 and -2.8 < jets[ll]["eta"] < 2.8    ]

    # ---- BTAGGED JETS ----
    index_btag = [
        ll for ll in index_jet     # btag using the jet subset
        if jets[ll]["btag"] == 1 and -2.5 < jets[ll]["eta"] < 2.5    ]

    return index_photon, index_e, index_mu, index_tau, index_jet, index_btag

## SELECTION CUTS

In [3]:
def selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag):
    """
    Applies specific cuts
    
    Input:
     - a single LHCO reader event
     - the indices of the particles that pass the basic id cuts
     
    Returns:
     - pass if the event satisfies the specific cuts
     - the index of the particles that satisfy the specific cuts:
     index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut
    """

    #######################
    # ----- LEPTONS ----- #
    #######################

    # Events containing isolated leptons with pT > 25 GeV and |η| < 2.5 are vetoed.

    # --- ELECTRON CUT --- #
    e_cut = True
    index_e_cut = []

    for ll_e in index_e:
        pt  = event["electron"][ll_e]["PT"]
        eta = event["electron"][ll_e]["eta"]

        if pt > 25 and -2.5 < eta < 2.5:
            e_cut = False
            break
        else:
            index_e_cut.append(ll_e)

    if not e_cut:
        return False, None, None, None, None, None, None   # early exit, skip the rest


    # --- MUON CUT --- #
    mu_cut = True
    index_mu_cut = []

    for ll_mu in index_mu:
        pt  = event["muon"][ll_mu]["PT"]
        eta = event["muon"][ll_mu]["eta"]

        if pt > 25 and -2.5 < eta < 2.5:
            mu_cut = False
            break
        else:
            index_mu_cut.append(ll_mu)

    if not mu_cut:
        return False, None, None, None, None, None, None   # early exit, skip the rest


    # --- TAU CUT --- #
    tau_cut = True
    index_tau_cut = []

    for ll_tau in index_tau:
        pt  = event["tau"][ll_tau]["PT"]
        eta = event["tau"][ll_tau]["eta"]

        if pt > 25 and -2.5 < eta < 2.5:
            tau_cut = False
            break
        else:
            index_tau_cut.append(ll_tau)

    if not tau_cut:
        return False, None, None, None, None, None, None   # early exit, skip the rest


    ###################
    # ----- JETs ---- #
    ###################

    # Events with more than five jets with |pT | > 30 GeV and |η| < 2.5 are vetoed.
    # This includes the tagged b-jets with |η| < 2.4

    jets = event["jet"]

    index_jet_cut = []
    num_good_jets = 0

    for ll_jet in index_jet:
        jet  = jets[ll_jet]
        pt   = jet["PT"]
        eta  = jet["eta"]
        btag = jet["btag"]

        # non-btag jets
        if btag == 0:
            good = (pt > 30 and -2.5 < eta < 2.5)
            if good:
                index_jet_cut.append(ll_jet)
                num_good_jets += 1

        # b-tagged jets
        else:
            good = (pt > 30 and -2.4 < eta < 2.4)
            if good:
                num_good_jets += 1

    if num_good_jets > 5:
        return False, None, None, None, None, None, None   # early exit, skip the rest


    ######################
    # ----- PHOTONS ---- #
    ######################

    # At least two isolated photons with pT > 30 GeV and |η| < 1.37 or 1.52 < |η| < 2.37.

    if len(index_photon) < 2:
        return False, None, None, None, None, None, None   # early exit, skip the rest

    index_photon_cut = []

    for ll_photon in index_photon:
        pt  = event["photon"][ll_photon]["PT"]
        eta = event["photon"][ll_photon]["eta"]

        good_pt = pt > 30
        good_eta = (
            -2.37 < eta < -1.52 or
            -1.37 < eta < 1.37 or
             1.52 < eta < 2.37
        )

        if good_pt and good_eta:
            index_photon_cut.append(ll_photon)

    if len(index_photon_cut) < 2:
        return False, None, None, None, None, None, None   # early exit, skip the rest


    #################################
    # ----- B-TAGGED JETS CUT ----- #
    #################################

    # At least two b-tagged jets with |η| < 2.4 and pT > 40 GeV and 30 GeV, for the leading and subleading b-jets, respectively

    if len(index_btag) < 2:
        return False, None, None, None, None, None, None   # early exit, skip the rest

    jets = event["jet"]

    index_btag_cut  = []
    index_lead_btag = []

    for ll_btag in index_btag:
        pt  = jets[ll_btag]["PT"]
        eta = jets[ll_btag]["eta"]

        if pt > 40 and -2.4 < eta < 2.4:
            index_lead_btag.append(ll_btag)

        if pt > 30 and -2.4 < eta < 2.4:
            index_btag_cut.append(ll_btag)

    if len(index_lead_btag) == 0:
        return False, None, None, None, None, None, None   # early exit, skip the rest

    if len(index_btag_cut) < 2:
        return False, None, None, None, None, None, None   # early exit, skip the rest


    ###########################
    #    ALL CUTS PASSED      #
    ###########################

    return True, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut


### Functions to store the selected events

In [4]:
# THIS ONES ARE TO SAVE IN .h5 FILES


# construct the vectors
dt = h5py.special_dtype(vlen=np.float32)

def ensure_vec3(name):
    """Creates name_pt, name_eta, name_phi if they do not exist."""
    for comp in ["pt", "eta", "phi"]:
        feature = "{}_{}".format(name, comp)
        if feature not in FileSave:
            FileSave.create_dataset(feature, shape=(0,), maxshape=(None,), dtype=dt)

def ensure_vec2(name):
    """Creates name_pt, name_phi if they do not exist (for MET basically)."""
    for comp in ["pt", "phi"]:
        feature = "{}_{}".format(name, comp)
        if feature not in FileSave:
            FileSave.create_dataset(feature, shape=(0,), maxshape=(None,), dtype=dt)

In [5]:
# to save the event
def save_event(FileSave, N_saved,
               photon_pts, photon_etas, photon_phis,
               btag_pts, btag_etas, btag_phis,
               jet_pts, jet_etas, jet_phis,
               MET_pts, MET_phis):
    """
    Saves the event

    Input:
     - FileSave: the .h5
     - N_saved: the place (number) where the event has to be saved (to avoid overwritting already saved data)
     - the features that you want to save

    Returns:
     - N_saved += 1
     - (the event saved is just in the .h5)
    """

    # --- HDF5 resize ---
    keys = [
        "photon_pt", "photon_eta", "photon_phi",
        "btag_pt", "btag_eta", "btag_phi",
        "jet_pt", "jet_eta", "jet_phi",
        "MET_pt", "MET_phi",
    ]

    for key in keys:
        FileSave[key].resize((N_saved+1,))

    # --- write event ---
    FileSave["photon_pt"][N_saved]  = np.array(photon_pts,  dtype=np.float32)
    FileSave["photon_eta"][N_saved] = np.array(photon_etas, dtype=np.float32)
    FileSave["photon_phi"][N_saved] = np.array(photon_phis, dtype=np.float32)

    FileSave["btag_pt"][N_saved]  = np.array(btag_pts,  dtype=np.float32)
    FileSave["btag_eta"][N_saved] = np.array(btag_etas, dtype=np.float32)
    FileSave["btag_phi"][N_saved] = np.array(btag_phis, dtype=np.float32)

    FileSave["jet_pt"][N_saved]  = np.array(jet_pts,  dtype=np.float32)
    FileSave["jet_eta"][N_saved] = np.array(jet_etas, dtype=np.float32)
    FileSave["jet_phi"][N_saved] = np.array(jet_phis, dtype=np.float32)

    FileSave["MET_pt"][N_saved]  = np.array(MET_pts,  dtype=np.float32)
    FileSave["MET_phi"][N_saved] = np.array(MET_phis, dtype=np.float32)

    return N_saved + 1


### Functions to get the cross-section computed with MG

In [6]:
# If we have an .lhco
def get_cross_section_from_lhco(path):
    """
    Get the cross section [pb] from a file .lhco
    Returns float or None if can't be found
    """
    pat_matched = re.compile(r"Matched Integrated weight \(pb\)\s*:\s*([0-9eE\.\+-]+)")
    pat_unmatched = re.compile(r"Integrated weight \(pb\)\s*:\s*([0-9eE\.\+-]+)")

    matched = None
    unmatched = None

    with open(path, "r") as f:
        for line in f:
            if matched is None:
                m = pat_matched.search(line)
                if m:
                    matched = float(m.group(1))
                    continue

            if unmatched is None:
                m = pat_unmatched.search(line)
                if m:
                    unmatched = float(m.group(1))
                    continue

    if matched is not None:
        return matched

    if unmatched is not None:
        print("[WARNING] Using non-matched Integrated weight (pb).")
        return unmatched

    return None


#  If the .lhco is compressed into a .gz
def get_cross_section_from_lhco_gz(path):
    """
    Get the cross section [pb] from a file .lhco.gz
    Returns float or None if can't be found
    """
    pat_matched = re.compile(r"Matched Integrated weight \(pb\)\s*:\s*([0-9eE\.\+-]+)")
    pat_unmatched = re.compile(r"Integrated weight \(pb\)\s*:\s*([0-9eE\.\+-]+)")

    matched = None
    unmatched = None

    with gzip.open(path, "rt") as f:
        for line in f:
            if matched is None:
                m = pat_matched.search(line)
                if m:
                    matched = float(m.group(1))
                    continue

            if unmatched is None:
                m = pat_unmatched.search(line)
                if m:
                    unmatched = float(m.group(1))
                    continue

    if matched is not None:
        return matched

    if unmatched is not None:
        print("[WARNING] Using non-matched Integrated weight (pb).")
        return unmatched

    return None

### Function to divide a huge .lhco file

In [7]:
# WORKS WITH BOTH .lhco and .lhco.gz FILES #

def split_lhco_auto_py2(path, out_dir, parts=10, prefix=None):
    """
    Divides a file .lhco or .lhco.gz (using low amounts of RAM)
    Saves the headers in a separated file
    compatible with Python 2.7.
    """

    # -----------------------------
    # Basic checks
    # -----------------------------
    # check if the input file exists
    if not os.path.isfile(path):
        raise RuntimeError("File not found: {}".format(path))

    # check if you defined a name for the outputs, otherwise copies the input name (without .gz  nor .lhco)
    if prefix is None:
        base = os.path.basename(path)
        base = base.replace(".gz", "")
        base = base.replace(".lhco", "")
        prefix = base

    # check if there is a folder for the output
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    # Detects if the input is a .gz file or not
    is_gz = path.endswith(".gz")

    # Function to open the file (either .lhco.gz or .lhco, works for both)
    def _open(file_path):
        return gzip.open(file_path, "rb") if is_gz else open(file_path, "r")

    # ---------------------------------
    # 1) Extract headers and count them
    # ---------------------------------
    header_lines = []
    total_events = 0
    inside_events = False

    f = _open(path)

    for raw_line in f:
        # Convert to stings if data is in bytes (should not happend)
        try:
            line = raw_line.decode("utf-8")
        except:
            line = raw_line

        stripped = line.lstrip()

        # search the begining of the events block
        if stripped.startswith("0 "):

            total_events += 1

            if not inside_events:
                inside_events = True  # 1st time we encounter an event

            # from now on, we dont save the header
        else:
            if not inside_events:
                header_lines.append(line)

    f.close()

    if total_events == 0:
        raise RuntimeError("No events found in file: {}".format(path))

    print("Total events found:", total_events)

    # Save the headers
    header_path = os.path.join(out_dir, prefix + "_headers.lhco")
    h = open(header_path, "w")
    for L in header_lines:
        # h.write(L)
        h.write(L.encode('utf-8', 'ignore'))
    h.close()
    print("Headers saved to:", header_path)

    # -----------------------------
    # 2) Now open the file again to divide it
    # -----------------------------
    events_per_part = (total_events + parts - 1) // parts

    # prepare the 1st file
    part_idx = 0
    ev_in_current = 0

    out_file = os.path.join(out_dir,
        "{}_part{:02d}.lhco".format(prefix, part_idx))
    fout = open(out_file, "w")
    print("Writing:", out_file)

    f = _open(path)

    inside_events = False

    for raw_line in f:
        try:
            line = raw_line.decode("utf-8")
        except:
            line = raw_line

        stripped = line.lstrip()

        # begining of the event
        if stripped.startswith("0 "):

            if not inside_events:
                inside_events = True

            # If this file is full, open a new one
            if ev_in_current >= events_per_part and part_idx < parts - 1:
                fout.close()
                part_idx += 1
                ev_in_current = 0

                out_file = os.path.join(out_dir,
                    "{}_part{:02d}.lhco".format(prefix, part_idx))
                fout = open(out_file, "w")
                print("Writing:", out_file)

            ev_in_current += 1

        # Only write events, not headers
        if inside_events:
            fout.write(line)

    f.close()
    fout.close()

    print("DONE.")

    # List of files generated
    part_files = [
        os.path.join(out_dir, "{}_part{:02d}.lhco".format(prefix, i))
        for i in range(parts)
    ]

    return header_path, part_files


# TEST: id cuts

In [10]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/'
FileName = 'tth_cluster_ale_'


ev = []


for i_run in range(1,2):
    
    datarun = LHCO_reader.Events(f_name=Folder+FileName+"run_0"+str(i_run)+".lhco")
    print(datarun)


    for i_ev, event in enumerate(datarun):

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        print('Event number: ', i_ev)

        print('\n   Type                  #        particle index')
        print('Photons             ', len(index_photon), '     ', index_photon)
        print('Electrons           ', len(index_e), '     ', index_e)
        print('Mu                  ', len(index_mu), '     ', index_mu)
        print('Taus                ', len(index_tau), '     ', index_tau)
        print('Jets (with b-jets)  ', len(index_jet), '     ', index_jet)
        print('b-jets              ', len(index_btag), '     ', index_btag)
        print('')

        print('--------------\n')


        if i_ev == 5:
            break
        

+------------------+----------------------------------------------------------------------------------------------------------------+
| Number of events | 66063                                                                                                          |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/tth_cluster_ale_run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------------------+
('Event number: ', 0)

   Type                  #        particle index
('Photons             ', 1, '     ', [0])
('Electrons           ', 0, '     ', [])
('Mu                  ', 0, '     ', [])
('Taus                ', 0, '     ', [])
('Jets (with b-jets)  ', 4, '     ', [0, 1, 2, 3])
('b-jets              ', 2, '     ', [0, 3])

--------------

('Event number: ', 1)

   Type                  #        particle index
('Photons             ', 1, '     ', [0])


/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


# ESPECIFIC CUTS (WORK SITE)

# BACKGROUNDS

## ttH

#### files like: tth_cluster_ale_

In [10]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/'
file_path = Folder + "tth_cluster_ale_run_01.lhco"

MG_cross_tth = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_tth, "[pb]")

('Cross section (MG) = ', 0.000781928119, '[pb]')


In [8]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/'
FileName = 'tth_cluster_ale_'

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/tth.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(1,119):

    file_path = Folder + FileName + "run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------------------+
| Number of events | 66063                                                                                                          |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/tth_cluster_ale_run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 66063/66063 [00:10<00:00, 6506.98it/s]


('run: ', 1, '   # events ok so far: ', 2922)
+------------------+----------------------------------------------------------------------------------------------------------------+
| Number of events | 66170                                                                                                          |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/tth_cluster_ale_run_02.lhco |
+------------------+----------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 66170/66170 [00:15<00:00, 4232.17it/s]


('run: ', 2, '   # events ok so far: ', 5806)
+------------------+----------------------------------------------------------------------------------------------------------------+
| Number of events | 66354                                                                                                          |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/tth_cluster_ale_run_03.lhco |
+------------------+----------------------------------------------------------------------------------------------------------------+


...

('run: ', 116, '   # events ok so far: ', 331126)
+------------------+-----------------------------------------------------------------------------------------------------------------+
| Number of events | 66037                                                                                                           |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/tth_cluster_ale_run_117.lhco |
+------------------+-----------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 66037/66037 [00:08<00:00, 7628.99it/s]


('run: ', 117, '   # events ok so far: ', 334022)
+------------------+-----------------------------------------------------------------------------------------------------------------+
| Number of events | 66105                                                                                                           |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/tth_cluster_ale_run_118.lhco |
+------------------+-----------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 66105/66105 [00:08<00:00, 7631.07it/s]

('run: ', 118, '   # events ok so far: ', 336892)
 
('Total initial events: ', 7800950)
('Total events after cuts: ', 336892)


In [9]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 7800950)
('Total events after cuts: ', 336892)


In [11]:
initial_evs = 7800950
events_ok = 336892

In [12]:
cross_fb = MG_cross_tth*1000 # 1pb = 1000 fb
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000  # fb^-1

fid_cross = cross_fb * aceptancia # fb
Tot_ev_expected = cross_fb * aceptancia * luminosity # fb * fb^-1

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.033768365117857184, '[fb]')
('Events expected: ', 101.30509535357155, '    for L=', 3000, ' [fb-1]')


#### files like: tth_pcleandro_ (.gz)

In [13]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/'
file_path = Folder + "tth_pcleandro_run_06.lhco.gz"

MG_cross_tth = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_tth, "[pb]")

('Cross section (MG) = ', 0.0007813907961875, '[pb]')


In [12]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/'
FileName = 'tth_pcleandro_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/tth.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(6, 50):  

    archive_path = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)

    if not os.path.isfile(archive_path):
        print("[WARNING] File not found: {} -> skipping".format(archive_path))
        continue

    # create a tmp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".lhco")
    tmp_path = tmp.name

    with gzip.open(archive_path, "rb") as gz, open(tmp_path, "wb") as out:
        out.write(gz.read())

    # read the LHCO
    datarun = LHCO_reader.Events(tmp_path)

   
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:361: UserWarning: Did not parse all events in file
  warnings.warn("Did not parse all events in file")


+------------------+---------------------+
| Number of events | 66018               |
| Description      | /tmp/tmpYXwiGd.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 66018/66018 [00:13<00:00, 4863.54it/s]


('run: ', 6, '   # events ok so far: ', 2898)
+------------------+---------------------+
| Number of events | 66111               |
| Description      | /tmp/tmpnhYY_3.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 66111/66111 [00:11<00:00, 5874.12it/s]


...

('run: ', 47, '   # events ok so far: ', 120228)
+------------------+---------------------+
| Number of events | 66056               |
| Description      | /tmp/tmphipNl8.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 66056/66056 [00:08<00:00, 7347.57it/s]


('run: ', 48, '   # events ok so far: ', 123110)
+------------------+---------------------+
| Number of events | 66021               |
| Description      | /tmp/tmpZt4yzh.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 66021/66021 [00:09<00:00, 7188.04it/s]

('run: ', 49, '   # events ok so far: ', 126008)
 
('Total initial events: ', 2909513)
('Total events after cuts: ', 126008)


In [13]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 2909513)
('Total events after cuts: ', 126008)


In [14]:
initial_evs = 2909513
events_ok = 126008

In [15]:
cross_fb = MG_cross_tth*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.03384122753395311, '[fb]')
('Events expected: ', 101.52368260185933, '    for L=', 3000, ' [fb-1]')


#### files like: tth+j_cluster (.gz)

In [16]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/'
file_path = Folder + "tth+j_cluster_MG5332_run_03.lhco.gz"

MG_cross_tth = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_tth, "[pb]")

('Cross section (MG) = ', 0.00078341617225, '[pb]')


##### files too big, lets divide them

In [20]:
# location of the input file
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/'
FileName = 'tth+j_cluster_MG5332_' 

# location of the output files
OutputFolder = Folder + 'DIVIDED_tth+j_cluster'

# number of output files
parts = 10


for i_run in range(3, 8):  

    archive = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)
    outdir = OutputFolder

    split_lhco_auto_py2(archive, outdir, parts=parts)

    print('\n DONE file: ', archive)
    print('\n --------------------- \n')
    

('Total events found:', 661489)
('Headers saved to:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/tth+j_cluster_MG5332_run_03_headers.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/tth+j_cluster_MG5332_run_03_part00.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/tth+j_cluster_MG5332_run_03_part01.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/tth+j_cluster_MG5332_run_03_part02.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/tth+j_cluster_MG5332_run_03_part03.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/tth+j_cluster_MG5332_run_03_part04

In [22]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/'
FileName = 'tth+j_cluster_MG5332_' 

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/tth.h5", "a")


# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(3,8):

    for i_part in range(0,10):

        file_path = Folder + FileName + "run_{:02d}".format(i_run) + "_part{:02d}.lhco".format(i_part)
    
        # If the file does not exist, skip and go to the next one
        if not os.path.isfile(file_path):
            print("[WARNING] File not found: {} → skipping".format(file_path))
            continue
            
        datarun = LHCO_reader.Events(f_name=file_path)
        
        # show file characteristics and number of generated events
        print(datarun)
        initial_evs += len(datarun)
    
    
        for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar
    
            #####################
            # BASIC PARTICLE ID #
            #####################
    
            index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)
    
    
            
            ############################
            # ESPECIFIC SELECTION CUTS #
            ############################
    
            passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)
    
            if not passed:
                continue # skip and go to the next event
                
    
    
            ##############
            # SAVE EVENT #
            ##############
    
            photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
            photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
            photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
            
            btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
            btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
            btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]
    
            jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
            jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
            jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]
    
            MET_pts  = [event["MET"][0]["PT"]]
            MET_phis = [event["MET"][0]["phi"]]
    
            # save the event
            N_saved = save_event(   FileSave, N_saved,
                                    photon_pts, photon_etas, photon_phis,
                                    btag_pts, btag_etas, btag_phis,
                                    jet_pts, jet_etas, jet_phis,
                                    MET_pts, MET_phis                       )
    
    
    
            # count the number of events that passed everything so far
            events_ok += 1
    
    
        pass # for the progress bar

        print('run: ',i_run, '   part:', i_part,  '   # events ok so far: ' , events_ok)
        

    # print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 66149                                                                                                                                            |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/tth+j_cluster_MG5332_run_03_part00.lhco |
+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 66149/66149 [00:09<00:00, 6736.63it/s]


('run: ', 3, '   part:', 0, '   # events ok so far: ', 2763)
+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 66149                                                                                                                                            |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/tth+j_cluster_MG5332_run_03_part01.lhco |
+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 66149/66149 [00:17<00:00, 3826.72it/s]


...

('run: ', 7, '   part:', 7, '   # events ok so far: ', 137375)
+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 66109                                                                                                                                            |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/tth+j_cluster_MG5332_run_07_part08.lhco |
+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 66109/66109 [00:18<00:00, 3585.45it/s]


('run: ', 7, '   part:', 8, '   # events ok so far: ', 140287)
+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 66102                                                                                                                                            |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ttH/DIVIDED_tth+j_cluster/tth+j_cluster_MG5332_run_07_part09.lhco |
+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 66102/66102 [00:17<00:00, 3767.84it/s]

('run: ', 7, '   part:', 9, '   # events ok so far: ', 143182)
 
('Total initial events: ', 3305456)
('Total events after cuts: ', 143182)


In [23]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 3305456)
('Total events after cuts: ', 143182)


In [17]:
initial_evs = 3305456
events_ok = 143182

In [18]:
cross_fb = MG_cross_tth*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.03393513463047141, '[fb]')
('Events expected: ', 101.80540389141423, '    for L=', 3000, ' [fb-1]')


## bbaa

#### files like: bbaa_cluster_MG5332_ (.gz)

In [37]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
file_path = Folder + "bbaa_cluster_MG5332_run_06.lhco.gz"

MG_cross_bbaa = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaa, "[pb]")

('Cross section (MG) = ', 0.1247308405, '[pb]')


In [9]:
# location of the input file
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
FileName = 'bbaa_cluster_MG5332_' 

# location of the output files
OutputFolder = Folder + 'DIVIDED_bbaa_cluster'

# number of output files
parts = 10


for i_run in range(6, 25):  

    archive = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)
    outdir = OutputFolder

    split_lhco_auto_py2(archive, outdir, parts=parts)

    print('\n DONE file: ', archive)
    print('\n --------------------- \n')

('Total events found:', 755830)
('Headers saved to:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/bbaa_cluster_MG5332_run_06_headers.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/bbaa_cluster_MG5332_run_06_part00.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/bbaa_cluster_MG5332_run_06_part01.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/bbaa_cluster_MG5332_run_06_part02.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/bbaa_cluster_MG5332_run_06_part03.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/bbaa_cluster_MG5332_run_06_part04.lhco'

In [38]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/'
FileName = 'bbaa_cluster_MG5332_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaa.h5", "a")


# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(6, 25): 

    for i_part in range(0,10):

        file_path = Folder + FileName + "run_{:02d}".format(i_run) + "_part{:02d}.lhco".format(i_part)
    
        # If the file does not exist, skip and go to the next one
        if not os.path.isfile(file_path):
            print("[WARNING] File not found: {} → skipping".format(file_path))
            continue
            
        datarun = LHCO_reader.Events(f_name=file_path)
        
        # show file characteristics and number of generated events
        print(datarun)
        initial_evs += len(datarun)
    
    
        for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar
    
            #####################
            # BASIC PARTICLE ID #
            #####################
    
            index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)
    
    
            
            ############################
            # ESPECIFIC SELECTION CUTS #
            ############################
    
            passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)
    
            if not passed:
                continue # skip and go to the next event
                
    
    
            ##############
            # SAVE EVENT #
            ##############
    
            photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
            photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
            photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
            
            btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
            btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
            btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]
    
            jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
            jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
            jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]
    
            MET_pts  = [event["MET"][0]["PT"]]
            MET_phis = [event["MET"][0]["phi"]]
    
            # save the event
            N_saved = save_event(   FileSave, N_saved,
                                    photon_pts, photon_etas, photon_phis,
                                    btag_pts, btag_etas, btag_phis,
                                    jet_pts, jet_etas, jet_phis,
                                    MET_pts, MET_phis                       )
    
    
    
            # count the number of events that passed everything so far
            events_ok += 1
    
    
        pass # for the progress bar

        print('run: ',i_run, '   part:', i_part,  '   # events ok so far: ' , events_ok)
        

    # print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75583                                                                                                                                           |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/bbaa_cluster_MG5332_run_06_part00.lhco |
+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75583/75583 [00:04<00:00, 15785.75it/s]


('run: ', 6, '   part:', 0, '   # events ok so far: ', 1267)
+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75583                                                                                                                                           |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/bbaa_cluster_MG5332_run_06_part01.lhco |
+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75583/75583 [00:08<00:00, 9057.15it/s] 


...

('run: ', 24, '   part:', 7, '   # events ok so far: ', 247383)
+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75694                                                                                                                                           |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/bbaa_cluster_MG5332_run_24_part08.lhco |
+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75694/75694 [00:04<00:00, 17907.08it/s]


('run: ', 24, '   part:', 8, '   # events ok so far: ', 248717)
+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75688                                                                                                                                           |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster/bbaa_cluster_MG5332_run_24_part09.lhco |
+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75688/75688 [00:04<00:00, 18846.98it/s]

('run: ', 24, '   part:', 9, '   # events ok so far: ', 249997)
 
('Total initial events: ', 14360287)
('Total events after cuts: ', 249997)


In [39]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 14360287)
('Total events after cuts: ', 249997)


In [17]:
initial_evs = 14360287
events_ok = 249997

In [40]:
cross_fb = MG_cross_bbaa*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 2.171428463266681, '[fb]')
('Events expected: ', 6514.285389800043, '    for L=', 3000, ' [fb-1]')


#### files like: bbaa_cluster_MG5332v4_ (.gz)

In [41]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
file_path = Folder + "bbaa_cluster_MG5332v4_run_02.lhco.gz"

MG_cross_bbaa = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaa, "[pb]")

('Cross section (MG) = ', 0.124564055, '[pb]')


In [36]:
# location of the input file
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
FileName = 'bbaa_cluster_MG5332v4_' 

# location of the output files
OutputFolder = Folder + 'DIVIDED_bbaa_cluster_v4'

# number of output files
parts = 10


for i_run in range(2, 12):  

    archive = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)
    outdir = OutputFolder

    split_lhco_auto_py2(archive, outdir, parts=parts)

    print('\n DONE file: ', archive)
    print('\n --------------------- \n')

('Total events found:', 755208)
('Headers saved to:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluster_MG5332v4_run_02_headers.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluster_MG5332v4_run_02_part00.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluster_MG5332v4_run_02_part01.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluster_MG5332v4_run_02_part02.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluster_MG5332v4_run_02_part03.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluste

In [42]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/'
FileName = 'bbaa_cluster_MG5332v4_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaa.h5", "a")


# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(2, 12): 

    for i_part in range(0,10):

        file_path = Folder + FileName + "run_{:02d}".format(i_run) + "_part{:02d}.lhco".format(i_part)
    
        # If the file does not exist, skip and go to the next one
        if not os.path.isfile(file_path):
            print("[WARNING] File not found: {} → skipping".format(file_path))
            continue
            
        datarun = LHCO_reader.Events(f_name=file_path)
        
        # show file characteristics and number of generated events
        print(datarun)
        initial_evs += len(datarun)
    
    
        for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar
    
            #####################
            # BASIC PARTICLE ID #
            #####################
    
            index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)
    
    
            
            ############################
            # ESPECIFIC SELECTION CUTS #
            ############################
    
            passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)
    
            if not passed:
                continue # skip and go to the next event
                
    
    
            ##############
            # SAVE EVENT #
            ##############
    
            photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
            photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
            photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
            
            btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
            btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
            btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]
    
            jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
            jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
            jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]
    
            MET_pts  = [event["MET"][0]["PT"]]
            MET_phis = [event["MET"][0]["phi"]]
    
            # save the event
            N_saved = save_event(   FileSave, N_saved,
                                    photon_pts, photon_etas, photon_phis,
                                    btag_pts, btag_etas, btag_phis,
                                    jet_pts, jet_etas, jet_phis,
                                    MET_pts, MET_phis                       )
    
    
    
            # count the number of events that passed everything so far
            events_ok += 1
    
    
        pass # for the progress bar

        print('run: ',i_run, '   part:', i_part,  '   # events ok so far: ' , events_ok)
        

    # print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75521                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluster_MG5332v4_run_02_part00.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75521/75521 [00:08<00:00, 9186.80it/s] 


('run: ', 2, '   part:', 0, '   # events ok so far: ', 1306)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75521                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluster_MG5332v4_run_02_part01.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75521/75521 [00:08<00:00, 9154.95it/s] 


('run: ', 2, '   part:', 1, '   # events ok so far: ', 2607)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75521                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluster_MG5332v4_run_02_part02.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+


...

('run: ', 11, '   part:', 7, '   # events ok so far: ', 128311)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75547                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluster_MG5332v4_run_11_part08.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75547/75547 [00:04<00:00, 17692.00it/s]


('run: ', 11, '   part:', 8, '   # events ok so far: ', 129645)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75544                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_cluster_v4/bbaa_cluster_MG5332v4_run_11_part09.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75544/75544 [00:05<00:00, 13849.16it/s]

('run: ', 11, '   part:', 9, '   # events ok so far: ', 130955)
 
('Total initial events: ', 7556282)
('Total events after cuts: ', 130955)


In [43]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 7556282)
('Total events after cuts: ', 130955)


In [17]:
initial_evs = 7556282
events_ok = 130955

In [44]:
cross_fb = MG_cross_bbaa*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 2.1587714463971834, '[fb]')
('Events expected: ', 6476.31433919155, '    for L=', 3000, ' [fb-1]')


#### files like: bbaa_MG5332_ (.gz)

In [8]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
file_path = Folder + "bbaa_MG5332_run_188.lhco.gz"

MG_cross_bbaa = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaa, "[pb]")

('Cross section (MG) = ', 0.124518487375, '[pb]')


In [11]:
# location of the input file
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
FileName = 'bbaa_MG5332_' 

# location of the output files
OutputFolder = Folder + 'DIVIDED_bbaa_MG5332'

# number of output files
parts = 10


for i_run in range(188, 191):

    archive = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)
    outdir = OutputFolder

    split_lhco_auto_py2(archive, outdir, parts=parts)

    print('\n DONE file: ', archive)
    print('\n --------------------- \n')

('Total events found:', 755197)
('Headers saved to:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/bbaa_MG5332_run_188_headers.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/bbaa_MG5332_run_188_part00.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/bbaa_MG5332_run_188_part01.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/bbaa_MG5332_run_188_part02.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/bbaa_MG5332_run_188_part03.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/bbaa_MG5332_run_188_part04.lhco')
('Writing:', '/home/andres/CompuTools/Programa

In [12]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/'
FileName = 'bbaa_MG5332_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaa.h5", "a")


# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(188, 191):

    for i_part in range(0,10):

        file_path = Folder + FileName + "run_{:02d}".format(i_run) + "_part{:02d}.lhco".format(i_part)
    
        # If the file does not exist, skip and go to the next one
        if not os.path.isfile(file_path):
            print("[WARNING] File not found: {} → skipping".format(file_path))
            continue
            
        datarun = LHCO_reader.Events(f_name=file_path)
        
        # show file characteristics and number of generated events
        print(datarun)
        initial_evs += len(datarun)
    
    
        for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar
    
            #####################
            # BASIC PARTICLE ID #
            #####################
    
            index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)
    
    
            
            ############################
            # ESPECIFIC SELECTION CUTS #
            ############################
    
            passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)
    
            if not passed:
                continue # skip and go to the next event
                
    
    
            ##############
            # SAVE EVENT #
            ##############
    
            photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
            photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
            photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
            
            btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
            btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
            btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]
    
            jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
            jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
            jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]
    
            MET_pts  = [event["MET"][0]["PT"]]
            MET_phis = [event["MET"][0]["phi"]]
    
            # save the event
            N_saved = save_event(   FileSave, N_saved,
                                    photon_pts, photon_etas, photon_phis,
                                    btag_pts, btag_etas, btag_phis,
                                    jet_pts, jet_etas, jet_phis,
                                    MET_pts, MET_phis                       )
    
    
    
            # count the number of events that passed everything so far
            events_ok += 1
    
    
        pass # for the progress bar

        print('run: ',i_run, '   part:', i_part,  '   # events ok so far: ' , events_ok)
        

    # print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75520                                                                                                                                   |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/bbaa_MG5332_run_188_part00.lhco |
+------------------+-----------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75520/75520 [00:06<00:00, 10831.35it/s]


('run: ', 188, '   part:', 0, '   # events ok so far: ', 1280)
+------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75520                                                                                                                                   |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/bbaa_MG5332_run_188_part01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75520/75520 [00:10<00:00, 7040.21it/s] 


...

('run: ', 190, '   part:', 7, '   # events ok so far: ', 36387)
+------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75547                                                                                                                                   |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/bbaa_MG5332_run_190_part08.lhco |
+------------------+-----------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75547/75547 [00:11<00:00, 6537.80it/s]


('run: ', 190, '   part:', 8, '   # events ok so far: ', 37726)
+------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75544                                                                                                                                   |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_bbaa_MG5332/bbaa_MG5332_run_190_part09.lhco |
+------------------+-----------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75544/75544 [00:05<00:00, 14681.97it/s]

('run: ', 190, '   part:', 9, '   # events ok so far: ', 39010)
 
('Total initial events: ', 2266799)
('Total events after cuts: ', 39010)


In [13]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 2266799)
('Total events after cuts: ', 39010)


In [14]:
initial_evs = 2266799
events_ok = 39010

In [15]:
cross_fb = MG_cross_bbaa*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 2.142874684742119, '[fb]')
('Events expected: ', 6428.624054226357, '    for L=', 3000, ' [fb-1]')


#### files like: pp-to-bbaa+bbaaj_SM_ (.gz)

In [16]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
file_path = Folder + "pp-to-bbaa+bbaaj_SM_run_191_tag_1_delphes-ATLAS-photon-efficiency_events.lhco.gz"

MG_cross_bbaa = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaa, "[pb]")

('Cross section (MG) = ', 0.124620221, '[pb]')


In [17]:
# location of the input file
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
FileName = 'pp-to-bbaa+bbaaj_SM_' 

# location of the output files
OutputFolder = Folder + 'DIVIDED_pp-to-bbaa+bbaaj_SM'

# number of output files
parts = 10


for i_run in range(191, 199):

    archive = Folder + FileName + "run_{:02d}_tag_1_delphes-ATLAS-photon-efficiency_events.lhco.gz".format(i_run)
    outdir = OutputFolder

    split_lhco_auto_py2(archive, outdir, parts=parts)

    print('\n DONE file: ', archive)
    print('\n --------------------- \n')

('Total events found:', 755354)
('Headers saved to:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_191_tag_1_delphes-ATLAS-photon-efficiency_events_headers.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_191_tag_1_delphes-ATLAS-photon-efficiency_events_part00.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_191_tag_1_delphes-ATLAS-photon-efficiency_events_part01.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_191_tag_1_delphes-ATLAS-photon-efficiency_events_part02.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_

In [18]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/'
FileName = 'pp-to-bbaa+bbaaj_SM_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaa.h5", "a")


# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(191, 199):

    for i_part in range(0,10):

        file_path = Folder + FileName + "run_{:02d}_tag_1_delphes-ATLAS-photon-efficiency_events".format(i_run) + "_part{:02d}.lhco".format(i_part)
    
        # If the file does not exist, skip and go to the next one
        if not os.path.isfile(file_path):
            print("[WARNING] File not found: {} → skipping".format(file_path))
            continue
            
        datarun = LHCO_reader.Events(f_name=file_path)
        
        # show file characteristics and number of generated events
        print(datarun)
        initial_evs += len(datarun)
    
    
        for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar
    
            #####################
            # BASIC PARTICLE ID #
            #####################
    
            index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)
    
    
            
            ############################
            # ESPECIFIC SELECTION CUTS #
            ############################
    
            passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)
    
            if not passed:
                continue # skip and go to the next event
                
    
    
            ##############
            # SAVE EVENT #
            ##############
    
            photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
            photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
            photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
            
            btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
            btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
            btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]
    
            jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
            jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
            jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]
    
            MET_pts  = [event["MET"][0]["PT"]]
            MET_phis = [event["MET"][0]["phi"]]
    
            # save the event
            N_saved = save_event(   FileSave, N_saved,
                                    photon_pts, photon_etas, photon_phis,
                                    btag_pts, btag_etas, btag_phis,
                                    jet_pts, jet_etas, jet_phis,
                                    MET_pts, MET_phis                       )
    
    
    
            # count the number of events that passed everything so far
            events_ok += 1
    
    
        pass # for the progress bar

        print('run: ',i_run, '   part:', i_part,  '   # events ok so far: ' , events_ok)
        

    # print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75536                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_191_tag_1_delphes-ATLAS-photon-efficiency_events_part00.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75536/75536 [00:11<00:00, 6440.90it/s]


('run: ', 191, '   part:', 0, '   # events ok so far: ', 1347)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75536                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_191_tag_1_delphes-ATLAS-photon-efficiency_events_part01.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75536/75536 [00:05<00:00, 12866.60it/s]


('run: ', 191, '   part:', 1, '   # events ok so far: ', 2697)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75536                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_191_tag_1_delphes-ATLAS-photon-efficiency_events_part02.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


...

('run: ', 198, '   part:', 7, '   # events ok so far: ', 102140)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75512                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_198_tag_1_delphes-ATLAS-photon-efficiency_events_part08.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75512/75512 [00:09<00:00, 8036.18it/s] 


('run: ', 198, '   part:', 8, '   # events ok so far: ', 103441)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75504                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_198_tag_1_delphes-ATLAS-photon-efficiency_events_part09.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75504/75504 [00:09<00:00, 7776.75it/s] 

('run: ', 198, '   part:', 9, '   # events ok so far: ', 104744)
 
('Total initial events: ', 6043889)
('Total events after cuts: ', 104744)


In [19]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 6043889)
('Total events after cuts: ', 104744)


In [20]:
initial_evs = 6043889
events_ok = 104744

In [21]:
cross_fb = MG_cross_bbaa*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 2.159738610094262, '[fb]')
('Events expected: ', 6479.215830282787, '    for L=', 3000, ' [fb-1]')


#### files like: pp-to-bbaa+bbaaj_SM_ (.gz)  CONTINUATION

In [8]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
file_path = Folder + "pp-to-bbaa+bbaaj_SM_run_205_tag_1_delphes-ATLAS-photon-efficiency_events.lhco.gz"

MG_cross_bbaa = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaa, "[pb]")

('Cross section (MG) = ', 0.12452908125, '[pb]')


In [21]:
# location of the input file
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
FileName = 'pp-to-bbaa+bbaaj_SM_' 

# location of the output files
OutputFolder = Folder + 'DIVIDED_pp-to-bbaa+bbaaj_SM'

# number of output files
parts = 10


# for i_run in range(199, 209): 
for i_run in range(201, 209):

    archive = Folder + FileName + "run_{:02d}_tag_1_delphes-ATLAS-photon-efficiency_events.lhco.gz".format(i_run)
    outdir = OutputFolder

    split_lhco_auto_py2(archive, outdir, parts=parts)

    print('\n DONE file: ', archive)
    print('\n --------------------- \n')

('Total events found:', 755301)
('Headers saved to:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_201_tag_1_delphes-ATLAS-photon-efficiency_events_headers.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_201_tag_1_delphes-ATLAS-photon-efficiency_events_part00.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_201_tag_1_delphes-ATLAS-photon-efficiency_events_part01.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_201_tag_1_delphes-ATLAS-photon-efficiency_events_part02.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_

In [22]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/'
FileName = 'pp-to-bbaa+bbaaj_SM_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaa.h5", "a")


# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(199, 209):

    for i_part in range(0,10):

        file_path = Folder + FileName + "run_{:02d}_tag_1_delphes-ATLAS-photon-efficiency_events".format(i_run) + "_part{:02d}.lhco".format(i_part)
    
        # If the file does not exist, skip and go to the next one
        if not os.path.isfile(file_path):
            print("[WARNING] File not found: {} → skipping".format(file_path))
            continue
            
        datarun = LHCO_reader.Events(f_name=file_path)
        
        # show file characteristics and number of generated events
        print(datarun)
        initial_evs += len(datarun)
    
    
        for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar
    
            #####################
            # BASIC PARTICLE ID #
            #####################
    
            index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)
    
    
            
            ############################
            # ESPECIFIC SELECTION CUTS #
            ############################
    
            passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)
    
            if not passed:
                continue # skip and go to the next event
                
    
    
            ##############
            # SAVE EVENT #
            ##############
    
            photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
            photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
            photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
            
            btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
            btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
            btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]
    
            jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
            jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
            jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]
    
            MET_pts  = [event["MET"][0]["PT"]]
            MET_phis = [event["MET"][0]["phi"]]
    
            # save the event
            N_saved = save_event(   FileSave, N_saved,
                                    photon_pts, photon_etas, photon_phis,
                                    btag_pts, btag_etas, btag_phis,
                                    jet_pts, jet_etas, jet_phis,
                                    MET_pts, MET_phis                       )
    
    
    
            # count the number of events that passed everything so far
            events_ok += 1
    
    
        pass # for the progress bar

        print('run: ',i_run, '   part:', i_part,  '   # events ok so far: ' , events_ok)
        

    # print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75585                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_199_tag_1_delphes-ATLAS-photon-efficiency_events_part00.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75585/75585 [00:09<00:00, 8159.94it/s] 


('run: ', 199, '   part:', 0, '   # events ok so far: ', 1359)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75585                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_199_tag_1_delphes-ATLAS-photon-efficiency_events_part01.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75585/75585 [00:08<00:00, 8652.54it/s] 


...

('run: ', 208, '   part:', 7, '   # events ok so far: ', 128393)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75599                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_208_tag_1_delphes-ATLAS-photon-efficiency_events_part08.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75599/75599 [00:10<00:00, 7077.42it/s]


('run: ', 208, '   part:', 8, '   # events ok so far: ', 129750)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75598                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_208_tag_1_delphes-ATLAS-photon-efficiency_events_part09.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75598/75598 [00:10<00:00, 7001.15it/s]

('run: ', 208, '   part:', 9, '   # events ok so far: ', 131067)
 
('Total initial events: ', 7557270)
('Total events after cuts: ', 131067)


In [23]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 7557270)
('Total events after cuts: ', 131067)


In [20]:
initial_evs = 7557270
events_ok = 131067

In [24]:
cross_fb = MG_cross_bbaa*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 2.1597287237578846, '[fb]')
('Events expected: ', 6479.186171273654, '    for L=', 3000, ' [fb-1]')


#### files like: pp-to-bbaa+bbaaj_SM_ (.gz)  CONTINUATION 2

In [8]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
file_path = Folder + "pp-to-bbaa+bbaaj_SM_run_212_tag_1_delphes-ATLAS-photon-efficiency_events.lhco.gz"

MG_cross_bbaa = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaa, "[pb]")

('Cross section (MG) = ', 0.12450820262500001, '[pb]')


In [9]:
# location of the input file
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/'
FileName = 'pp-to-bbaa+bbaaj_SM_' 

# location of the output files
OutputFolder = Folder + 'DIVIDED_pp-to-bbaa+bbaaj_SM'

# number of output files
parts = 10


for i_run in range(209, 215):

    archive = Folder + FileName + "run_{:02d}_tag_1_delphes-ATLAS-photon-efficiency_events.lhco.gz".format(i_run)
    outdir = OutputFolder

    split_lhco_auto_py2(archive, outdir, parts=parts)

    print('\n DONE file: ', archive)
    print('\n --------------------- \n')

('Total events found:', 755586)
('Headers saved to:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_209_tag_1_delphes-ATLAS-photon-efficiency_events_headers.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_209_tag_1_delphes-ATLAS-photon-efficiency_events_part00.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_209_tag_1_delphes-ATLAS-photon-efficiency_events_part01.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_209_tag_1_delphes-ATLAS-photon-efficiency_events_part02.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_

In [10]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/'
FileName = 'pp-to-bbaa+bbaaj_SM_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaa.h5", "a")


# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(209, 215):

    for i_part in range(0,10):

        file_path = Folder + FileName + "run_{:02d}_tag_1_delphes-ATLAS-photon-efficiency_events".format(i_run) + "_part{:02d}.lhco".format(i_part)
    
        # If the file does not exist, skip and go to the next one
        if not os.path.isfile(file_path):
            print("[WARNING] File not found: {} → skipping".format(file_path))
            continue
            
        datarun = LHCO_reader.Events(f_name=file_path)
        
        # show file characteristics and number of generated events
        print(datarun)
        initial_evs += len(datarun)
    
    
        for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar
    
            #####################
            # BASIC PARTICLE ID #
            #####################
    
            index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)
    
    
            
            ############################
            # ESPECIFIC SELECTION CUTS #
            ############################
    
            passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)
    
            if not passed:
                continue # skip and go to the next event
                
    
    
            ##############
            # SAVE EVENT #
            ##############
    
            photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
            photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
            photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
            
            btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
            btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
            btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]
    
            jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
            jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
            jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]
    
            MET_pts  = [event["MET"][0]["PT"]]
            MET_phis = [event["MET"][0]["phi"]]
    
            # save the event
            N_saved = save_event(   FileSave, N_saved,
                                    photon_pts, photon_etas, photon_phis,
                                    btag_pts, btag_etas, btag_phis,
                                    jet_pts, jet_etas, jet_phis,
                                    MET_pts, MET_phis                       )
    
    
    
            # count the number of events that passed everything so far
            events_ok += 1
    
    
        pass # for the progress bar

        print('run: ',i_run, '   part:', i_part,  '   # events ok so far: ' , events_ok)
        

    # print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75559                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_209_tag_1_delphes-ATLAS-photon-efficiency_events_part00.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75559/75559 [00:06<00:00, 11786.43it/s]


('run: ', 209, '   part:', 0, '   # events ok so far: ', 1265)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75559                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_209_tag_1_delphes-ATLAS-photon-efficiency_events_part01.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75559/75559 [00:08<00:00, 9046.18it/s] 


...

('run: ', 214, '   part:', 7, '   # events ok so far: ', 75867)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75560                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_214_tag_1_delphes-ATLAS-photon-efficiency_events_part08.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75560/75560 [00:08<00:00, 9067.15it/s] 


('run: ', 214, '   part:', 8, '   # events ok so far: ', 77112)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 75560                                                                                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaa/DIVIDED_pp-to-bbaa+bbaaj_SM/pp-to-bbaa+bbaaj_SM_run_214_tag_1_delphes-ATLAS-photon-efficiency_events_part09.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 75560/75560 [00:09<00:00, 8369.41it/s] 

('run: ', 214, '   part:', 9, '   # events ok so far: ', 78437)
 
('Total initial events: ', 4532944)
('Total events after cuts: ', 78437)


In [11]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 4532944)
('Total events after cuts: ', 78437)


In [12]:
initial_evs = 4532944
events_ok = 78437

In [13]:
cross_fb = MG_cross_bbaa*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 2.1544607410321253, '[fb]')
('Events expected: ', 6463.382223096376, '    for L=', 3000, ' [fb-1]')


## ZH

In [19]:
# cross section (pb) WITHOUT MATCHING
MG_cross_zh_NOMATCHING = 0.27649831

#### files like: zh_pcleandro_ (.gz)

In [20]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ZH/'
file_path = Folder + "zh_pcleandro_run_02.lhco.gz"

MG_cross_zh = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_zh, "[pb]")

('Cross section (MG) = ', 0.0005348820827500001, '[pb]')


In [17]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ZH/'
FileName = 'zh_pcleandro_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/zh.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(2, 47):  

    archive_path = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)

    if not os.path.isfile(archive_path):
        print("[WARNING] File not found: {} -> skipping".format(archive_path))
        continue

    # create a tmp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".lhco")
    tmp_path = tmp.name

    with gzip.open(archive_path, "rb") as gz, open(tmp_path, "wb") as out:
        out.write(gz.read())

    # read the LHCO
    datarun = LHCO_reader.Events(tmp_path)

   
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+---------------------+
| Number of events | 69325               |
| Description      | /tmp/tmpMPIBkA.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 69325/69325 [00:04<00:00, 15847.50it/s]


('run: ', 2, '   # events ok so far: ', 841)
+------------------+---------------------+
| Number of events | 69292               |
| Description      | /tmp/tmpKx9PkL.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 69292/69292 [00:09<00:00, 7355.26it/s] 


('run: ', 3, '   # events ok so far: ', 1651)
+------------------+---------------------+
| Number of events | 69251               |
| Description      | /tmp/tmpJmfuvo.lhco |
+------------------+---------------------+


...

('run: ', 44, '   # events ok so far: ', 36317)
+------------------+---------------------+
| Number of events | 69373               |
| Description      | /tmp/tmpW2CVMF.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 69373/69373 [00:04<00:00, 15630.52it/s]


('run: ', 45, '   # events ok so far: ', 37169)
+------------------+---------------------+
| Number of events | 69167               |
| Description      | /tmp/tmp13ref9.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 69167/69167 [00:03<00:00, 21288.33it/s]

('run: ', 46, '   # events ok so far: ', 38013)
 
('Total initial events: ', 3117598)
('Total events after cuts: ', 38013)


In [18]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 3117598)
('Total events after cuts: ', 38013)


In [21]:
initial_evs = 3117598
events_ok = 38013

In [22]:
cross_fb = MG_cross_zh*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.006521839124728637, '[fb]')
('Events expected: ', 19.565517374185912, '    for L=', 3000, ' [fb-1]')


#### files like: zh_pcleandro_

In [23]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ZH/'
file_path = Folder + "zh_pcleandro_run_47.lhco"

MG_cross_zh = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_zh, "[pb]")

('Cross section (MG) = ', 0.000536761018, '[pb]')


In [21]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ZH/'
FileName = 'zh_pcleandro_'

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/zh.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(47,100):

    file_path = Folder + FileName + "run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+------------------------------------------------------------------------------------------------------------+
| Number of events | 69620                                                                                                      |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ZH/zh_pcleandro_run_47.lhco |
+------------------+------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 69620/69620 [00:03<00:00, 21906.87it/s]


('run: ', 47, '   # events ok so far: ', 837)
+------------------+------------------------------------------------------------------------------------------------------------+
| Number of events | 69385                                                                                                      |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ZH/zh_pcleandro_run_48.lhco |
+------------------+------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 69385/69385 [00:03<00:00, 21208.48it/s]


('run: ', 48, '   # events ok so far: ', 1694)
+------------------+------------------------------------------------------------------------------------------------------------+
| Number of events | 69443                                                                                                      |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ZH/zh_pcleandro_run_49.lhco |
+------------------+------------------------------------------------------------------------------------------------------------+


...

('run: ', 97, '   # events ok so far: ', 43352)
+------------------+------------------------------------------------------------------------------------------------------------+
| Number of events | 69497                                                                                                      |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ZH/zh_pcleandro_run_98.lhco |
+------------------+------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 69497/69497 [00:03<00:00, 21726.65it/s]


('run: ', 98, '   # events ok so far: ', 44204)
+------------------+------------------------------------------------------------------------------------------------------------+
| Number of events | 69556                                                                                                      |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/ZH/zh_pcleandro_run_99.lhco |
+------------------+------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 69556/69556 [00:03<00:00, 21613.78it/s]

('run: ', 99, '   # events ok so far: ', 45034)
 
('Total initial events: ', 3674463)
('Total events after cuts: ', 45034)


In [22]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 3674463)
('Total events after cuts: ', 45034)


In [24]:
initial_evs = 3674463
events_ok = 45034

In [25]:
cross_fb = MG_cross_zh*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.006578511114307587, '[fb]')
('Events expected: ', 19.73553334292276, '    for L=', 3000, ' [fb-1]')


## bbaj (without fakes)

#### files like: bbaj_cluster_MG5332_ (.gz)   runs 1 to 50

In [26]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
file_path = Folder + "bbaj_cluster_MG5332_run_01.lhco.gz"

MG_cross_bbaj = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaj, "[pb]")

('Cross section (MG) = ', 268.12362325, '[pb]')


In [25]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
FileName = 'bbaj_cluster_MG5332_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaj_noFAKES.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(1, 51):  

    archive_path = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)

    if not os.path.isfile(archive_path):
        print("[WARNING] File not found: {} -> skipping".format(archive_path))
        continue

    # create a tmp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".lhco")
    tmp_path = tmp.name

    with gzip.open(archive_path, "rb") as gz, open(tmp_path, "wb") as out:
        out.write(gz.read())

    # read the LHCO
    datarun = LHCO_reader.Events(tmp_path)

   
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+---------------------+
| Number of events | 617711              |
| Description      | /tmp/tmpEAEv5e.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 617711/617711 [00:05<00:00, 107442.39it/s]


('run: ', 1, '   # events ok so far: ', 33)
+------------------+---------------------+
| Number of events | 618011              |
| Description      | /tmp/tmp0I65xd.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 618011/618011 [00:10<00:00, 59630.86it/s] 


('run: ', 2, '   # events ok so far: ', 66)
+------------------+---------------------+
| Number of events | 617942              |
| Description      | /tmp/tmpsRX7Mm.lhco |
+------------------+---------------------+


...

('run: ', 48, '   # events ok so far: ', 1855)
+------------------+---------------------+
| Number of events | 618353              |
| Description      | /tmp/tmpZBRfwm.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 618353/618353 [00:06<00:00, 100974.27it/s]


('run: ', 49, '   # events ok so far: ', 1902)
+------------------+---------------------+
| Number of events | 617620              |
| Description      | /tmp/tmpg1ydq8.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 617620/617620 [00:05<00:00, 103517.07it/s]

('run: ', 50, '   # events ok so far: ', 1934)
 
('Total initial events: ', 30884022)
('Total events after cuts: ', 1934)


In [26]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 30884022)
('Total events after cuts: ', 1934)


In [27]:
initial_evs = 30884022
events_ok = 1934

In [28]:
cross_fb = MG_cross_bbaj*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 16.790270624904363, '[fb]')
('Events expected: ', 50370.81187471309, '    for L=', 3000, ' [fb-1]')


#### files like: bbaj_cluster_MG5332_ (.gz)   runs 1101 to 1123

In [29]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
file_path = Folder + "bbaj_cluster_MG5332_run_1101.lhco.gz"

MG_cross_bbaj = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaj, "[pb]")

('Cross section (MG) = ', 266.63051800000005, '[pb]')


In [29]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
FileName = 'bbaj_cluster_MG5332_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaj_noFAKES.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(1101, 1124):  

    archive_path = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)

    if not os.path.isfile(archive_path):
        print("[WARNING] File not found: {} -> skipping".format(archive_path))
        continue

    # create a tmp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".lhco")
    tmp_path = tmp.name

    with gzip.open(archive_path, "rb") as gz, open(tmp_path, "wb") as out:
        out.write(gz.read())

    # read the LHCO
    datarun = LHCO_reader.Events(tmp_path)

   
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+---------------------+
| Number of events | 61740               |
| Description      | /tmp/tmpzgD6kG.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 61740/61740 [00:01<00:00, 49841.21it/s]


('run: ', 1101, '   # events ok so far: ', 2)
+------------------+---------------------+
| Number of events | 61990               |
| Description      | /tmp/tmpG3o26u.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 61990/61990 [00:00<00:00, 84366.31it/s]


('run: ', 1102, '   # events ok so far: ', 5)
+------------------+---------------------+
| Number of events | 61909               |
| Description      | /tmp/tmppRwdO_.lhco |
+------------------+---------------------+


...

('run: ', 1121, '   # events ok so far: ', 77)
+------------------+---------------------+
| Number of events | 61625               |
| Description      | /tmp/tmp75VL4g.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 61625/61625 [00:00<00:00, 106004.17it/s]


('run: ', 1122, '   # events ok so far: ', 79)
+------------------+---------------------+
| Number of events | 61685               |
| Description      | /tmp/tmpViBa8a.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 61685/61685 [00:00<00:00, 102736.91it/s]

('run: ', 1123, '   # events ok so far: ', 82)
 
('Total initial events: ', 1422571)
('Total events after cuts: ', 82)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 1422571)
('Total events after cuts: ', 82)


In [30]:
initial_evs = 1422571
events_ok = 82

In [31]:
cross_fb = MG_cross_bbaj*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 15.369146760337447, '[fb]')
('Events expected: ', 46107.44028101234, '    for L=', 3000, ' [fb-1]')


#### files like: bbaj_pc_MG5332_ (.gz)

In [32]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
file_path = Folder + "bbaj_pc_MG5332_run_237.lhco.gz"

MG_cross_bbaj = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaj, "[pb]")

('Cross section (MG) = ', 267.99852462499996, '[pb]')


In [33]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
FileName = 'bbaj_pc_MG5332_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaj_noFAKES.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(237, 247):  

    archive_path = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)

    if not os.path.isfile(archive_path):
        print("[WARNING] File not found: {} -> skipping".format(archive_path))
        continue

    # create a tmp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".lhco")
    tmp_path = tmp.name

    with gzip.open(archive_path, "rb") as gz, open(tmp_path, "wb") as out:
        out.write(gz.read())

    # read the LHCO
    datarun = LHCO_reader.Events(tmp_path)

   
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+---------------------+
| Number of events | 617375              |
| Description      | /tmp/tmpD_RlTF.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 617375/617375 [00:06<00:00, 95209.05it/s] 


('run: ', 237, '   # events ok so far: ', 45)
+------------------+---------------------+
| Number of events | 618006              |
| Description      | /tmp/tmpEExTNU.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 618006/618006 [00:05<00:00, 112730.28it/s]


('run: ', 238, '   # events ok so far: ', 83)
+------------------+---------------------+
| Number of events | 617723              |
| Description      | /tmp/tmpD80GIj.lhco |
+------------------+---------------------+


...

('run: ', 244, '   # events ok so far: ', 337)
+------------------+---------------------+
| Number of events | 618088              |
| Description      | /tmp/tmp0JP6vZ.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 618088/618088 [00:05<00:00, 106473.02it/s]


('run: ', 245, '   # events ok so far: ', 380)
+------------------+---------------------+
| Number of events | 617115              |
| Description      | /tmp/tmpRQKW_5.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 617115/617115 [00:05<00:00, 103912.63it/s]

('run: ', 246, '   # events ok so far: ', 412)
 
('Total initial events: ', 6177657)
('Total events after cuts: ', 412)


In [34]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 6177657)
('Total events after cuts: ', 412)


In [33]:
initial_evs = 6177657
events_ok = 412

In [34]:
cross_fb = MG_cross_bbaj*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 17.87334456178127, '[fb]')
('Events expected: ', 53620.03368534381, '    for L=', 3000, ' [fb-1]')


## Include JET->PHOTON misidentification in bbaj

In [57]:
import random

fake_rate_jet_to_photon = 5e-4   # jet → photon probability


####################################
# MISIDENTIFICATION (JET → PHOTON) #
####################################

def jet_to_photon_misID(event, fake_rate):
    """
    Given an LHCO 'event' (from LHCO_reader), applies jet→photon misidentification.
    Returns a NEW event (LHCO style) with modified photon and jet lists.
    All jets (including b-tag) are allowed to fake.
    """
    
    # Copy objects (convert to dict)
    photons = [dict(p) for p in event["photon"]]
    jets    = [dict(j) for j in event["jet"]]

    jets_to_remove = []

    for j in range(len(jets)):
        if random.random() < fake_rate:

            # create new fake photon
            photons.append({
                "PT":  jets[j]["PT"],
                "eta": jets[j]["eta"],
                "phi": jets[j]["phi"]
            })

            jets_to_remove.append(j)

    # remove faked jets
    for idx in sorted(jets_to_remove, reverse=True):
        del jets[idx]

    # reorder photons by pT (LHCO convention)
    photons.sort(key=lambda p: p["PT"], reverse=True)

    # new event
    return {
        "photon": photons,
        "jet": jets,
        "electron": event["electron"],
        "muon": event["muon"],
        "tau": event["tau"],
        "MET": event["MET"]
    }


#################
# WEIGHT METHOD #
#################
def weight_per_event(event, fake_rate, basic_id_cuts, selection_cuts):
    """
    For each jet in the event:
        - force it to become a fake photon
        - check if the resulting event_j_mod passes cuts:
             eps[j] = 1 if it passes
             eps[j] = 0 if it DOES NOT pass
             
    Return the event weight = fake_rate * sum(eps)
    """

    jets = [dict(j) for j in event["jet"]]
    photons_original = [dict(p) for p in event["photon"]]

    eps = []  # acceptance per jet
    
    for j in range(len(jets)):

        # copy fresh lists
        photons = list(photons_original)
        new_jets = []

        for k in range(len(jets)):
            if k == j:
                # FORCE jet j to become a photon
                photons.append({
                    "PT":  jets[k]["PT"],
                    "eta": jets[k]["eta"],
                    "phi": jets[k]["phi"]
                })
            else:
                # all other jets remain unchanged
                new_jets.append(jets[k])

        photons.sort(key=lambda p: p["PT"], reverse=True)

        # build modified event
        event_j_mod = {
            "photon": photons,
            "jet": new_jets,
            "electron": event["electron"],
            "muon": event["muon"],
            "tau": event["tau"],
            "MET": event["MET"]
        }

        # now apply cuts
        idx_photon, idx_e, idx_mu, idx_tau, idx_jet, idx_btag = basic_id_cuts(event_j_mod)
        cuts_output = selection_cuts(event_j_mod, idx_photon, idx_e, idx_mu, idx_tau, idx_jet, idx_btag)

        passed = cuts_output[0]
        
        eps.append(1 if passed else 0)

    return fake_rate * sum(eps)


## bbaj (WITH fakes)

#### files like: bbaj_cluster_MG5332_ (.gz)   runs 1 to 50

In [53]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
file_path = Folder + "bbaj_cluster_MG5332_run_01.lhco.gz"

MG_cross_bbaj = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaj, "[pb]")

('Cross section (MG) = ', 268.12362325, '[pb]')


In [47]:
# location of the input file
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
FileName = 'bbaj_cluster_MG5332_'   

# location of the output files
OutputFolder = Folder + 'DIVIDE_bbaj_cluster'

# number of output files
parts = 4


for i_run in range(1, 51): 

    archive = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)
    outdir = OutputFolder

    split_lhco_auto_py2(archive, outdir, parts=parts)

    print('\n DONE file: ', archive)
    print('\n --------------------- \n')
    

('Total events found:', 617711)
('Headers saved to:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/bbaj_cluster_MG5332_run_01_headers.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/bbaj_cluster_MG5332_run_01_part00.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/bbaj_cluster_MG5332_run_01_part01.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/bbaj_cluster_MG5332_run_01_part02.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/bbaj_cluster_MG5332_run_01_part03.lhco')
DONE.
('\n DONE file: ', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/bbaj_cluster_MG5332_run_01.lhco.gz')

 --------------

In [64]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/'
FileName = 'bbaj_cluster_MG5332_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaj_wFAKES.h5", "a")


# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0
events_ok_weight_method = 0


for i_run in range(1, 51): 
# for i_run in range(1, 3): 

    for i_part in range(0,4):

        file_path = Folder + FileName + "run_{:02d}".format(i_run) + "_part{:02d}.lhco".format(i_part)
    
        # If the file does not exist, skip and go to the next one
        if not os.path.isfile(file_path):
            print("[WARNING] File not found: {} → skipping".format(file_path))
            continue
            
        datarun = LHCO_reader.Events(f_name=file_path)
        
        # show file characteristics and number of generated events
        print(datarun)
        initial_evs += len(datarun)
    
    
        for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

            #################
            # WEIGHT METHOD #
            #################

            weight_ev = weight_per_event(event, fake_rate_jet_to_photon, basic_id_cuts, selection_cuts)

            # count the number of expected events that passed everything with the weight method
            events_ok_weight_method += weight_ev


            
            ####################################
            # MISIDENTIFICATION (JET → PHOTON) #
            ####################################
            
            event = jet_to_photon_misID(event, fake_rate_jet_to_photon)

            
    
            #####################
            # BASIC PARTICLE ID #
            #####################
    
            index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)
    
    
            
            ############################
            # ESPECIFIC SELECTION CUTS #
            ############################
    
            passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)
    
            if not passed:
                continue # skip and go to the next event
                
    
    
            ##############
            # SAVE EVENT #
            ##############
    
            photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
            photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
            photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
            
            btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
            btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
            btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]
    
            jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
            jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
            jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]
    
            MET_pts  = [event["MET"][0]["PT"]]
            MET_phis = [event["MET"][0]["phi"]]
    
            # save the event
            N_saved = save_event(   FileSave, N_saved,
                                    photon_pts, photon_etas, photon_phis,
                                    btag_pts, btag_etas, btag_phis,
                                    jet_pts, jet_etas, jet_phis,
                                    MET_pts, MET_phis                       )
    
    
    
            # count the number of events that passed everything so far
            events_ok += 1
    
    
        pass # for the progress bar

        print('run: ',i_run, '   part:', i_part,  '   # events ok so far: ' , events_ok)
        

    # print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)
print('Total events after cuts (weights): ', events_ok_weight_method)

+------------------+------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 154428                                                                                                                                         |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/bbaj_cluster_MG5332_run_01_part00.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 154428/154428 [00:06<00:00, 23530.75it/s]


('run: ', 1, '   part:', 0, '   # events ok so far: ', 8)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 154428                                                                                                                                         |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/bbaj_cluster_MG5332_run_01_part01.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 154428/154428 [00:05<00:00, 26994.34it/s]


('run: ', 1, '   part:', 1, '   # events ok so far: ', 19)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 154428                                                                                                                                         |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/bbaj_cluster_MG5332_run_01_part02.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------+


...

('run: ', 50, '   part:', 1, '   # events ok so far: ', 2210)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 154405                                                                                                                                         |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/bbaj_cluster_MG5332_run_50_part02.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 154405/154405 [00:03<00:00, 40873.76it/s]


('run: ', 50, '   part:', 2, '   # events ok so far: ', 2218)
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 154405                                                                                                                                         |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_cluster/bbaj_cluster_MG5332_run_50_part03.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 154405/154405 [00:03<00:00, 40669.67it/s]


('run: ', 50, '   part:', 3, '   # events ok so far: ', 2233)
 
('Total initial events: ', 30884022)
('Total events after cuts: ', 2233)
('Total events after cuts (weights): ', 290.6909999986472)


In [65]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)
print('Total events after cuts (weights): ', events_ok_weight_method)

('Total initial events: ', 30884022)
('Total events after cuts: ', 2233)
('Total events after cuts (weights): ', 290.6909999986472)


In [27]:
initial_evs = 30884022
events_ok = 2233
events_ok_weight_method = 290.6909999986472

In [66]:
cross_fb = MG_cross_bbaj*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")


aceptancia_W = 1.*events_ok_weight_method/initial_evs

fid_cross_W = cross_fb * aceptancia_W
Tot_ev_expected_W = cross_fb * aceptancia_W * luminosity

print('')
print('Fiducial cross section (weight method): ', fid_cross_W, "[fb]")
print('Events expected (weight method): ', Tot_ev_expected_W, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 19.386077717379234, '[fb]')
('Events expected: ', 58158.2331521377, '    for L=', 3000, ' [fb-1]')

('Fiducial cross section (weight method): ', 2.5236714365053565, '[fb]')
('Events expected (weight method): ', 7571.014309516069, '    for L=', 3000, ' [fb-1]')


#### files like: bbaj_cluster_MG5332_ (.gz)   runs 1101 to 1123

In [67]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
file_path = Folder + "bbaj_cluster_MG5332_run_1101.lhco.gz"

MG_cross_bbaj = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaj, "[pb]")

('Cross section (MG) = ', 266.63051800000005, '[pb]')


In [68]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
FileName = 'bbaj_cluster_MG5332_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaj_wFAKES.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0
events_ok_weight_method = 0


for i_run in range(1101, 1124):  

    archive_path = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)

    if not os.path.isfile(archive_path):
        print("[WARNING] File not found: {} -> skipping".format(archive_path))
        continue

    # create a tmp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".lhco")
    tmp_path = tmp.name

    with gzip.open(archive_path, "rb") as gz, open(tmp_path, "wb") as out:
        out.write(gz.read())

    # read the LHCO
    datarun = LHCO_reader.Events(tmp_path)

   
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #################
        # WEIGHT METHOD #
        #################

        weight_ev = weight_per_event(event, fake_rate_jet_to_photon, basic_id_cuts, selection_cuts)

        # count the number of expected events that passed everything with the weight method
        events_ok_weight_method += weight_ev


        
        ####################################
        # MISIDENTIFICATION (JET → PHOTON) #
        ####################################
        
        event = jet_to_photon_misID(event, fake_rate_jet_to_photon)

        

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)
print('Total events after cuts (weights): ', events_ok_weight_method)

+------------------+---------------------+
| Number of events | 61740               |
| Description      | /tmp/tmpNdRT3Y.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 61740/61740 [00:01<00:00, 37567.75it/s]


('run: ', 1101, '   # events ok so far: ', 4)
+------------------+---------------------+
| Number of events | 61990               |
| Description      | /tmp/tmpjizm00.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 61990/61990 [00:01<00:00, 39599.07it/s]


('run: ', 1102, '   # events ok so far: ', 7)
+------------------+---------------------+
| Number of events | 61909               |
| Description      | /tmp/tmp427aKC.lhco |
+------------------+---------------------+


...

('run: ', 1121, '   # events ok so far: ', 88)
+------------------+---------------------+
| Number of events | 61625               |
| Description      | /tmp/tmpCwN86i.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 61625/61625 [00:01<00:00, 42442.65it/s]


('run: ', 1122, '   # events ok so far: ', 90)
+------------------+---------------------+
| Number of events | 61685               |
| Description      | /tmp/tmpdKOFQh.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 61685/61685 [00:01<00:00, 40302.24it/s]

('run: ', 1123, '   # events ok so far: ', 93)
 
('Total initial events: ', 1422571)
('Total events after cuts: ', 93)
('Total events after cuts (weights): ', 13.333000000001524)


In [69]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)
print('Total events after cuts (weights): ', events_ok_weight_method)

('Total initial events: ', 1422571)
('Total events after cuts: ', 93)
('Total events after cuts (weights): ', 13.333000000001524)


In [30]:
initial_evs = 1422571
events_ok = 93
events_ok_weight_method = 13.333000000001524

In [70]:
cross_fb = MG_cross_bbaj*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")


aceptancia_W = 1.*events_ok_weight_method/initial_evs

fid_cross_W = cross_fb * aceptancia_W
Tot_ev_expected_W = cross_fb * aceptancia_W * luminosity

print('')
print('Fiducial cross section (weight method): ', fid_cross_W, "[fb]")
print('Events expected (weight method): ', Tot_ev_expected_W, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 17.430861569651007, '[fb]')
('Events expected: ', 52292.58470895302, '    for L=', 3000, ' [fb-1]')

('Fiducial cross section (weight method): ', 2.498985777507349, '[fb]')
('Events expected (weight method): ', 7496.957332522047, '    for L=', 3000, ' [fb-1]')


#### files like: bbaj_pc_MG5332_ (.gz)

In [71]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
file_path = Folder + "bbaj_pc_MG5332_run_237.lhco.gz"

MG_cross_bbaj = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_bbaj, "[pb]")

('Cross section (MG) = ', 267.99852462499996, '[pb]')


In [49]:
# location of the input file
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/'
FileName = 'bbaj_pc_MG5332_' 

# location of the output files
OutputFolder = Folder + 'DIVIDE_bbaj_pc'

# number of output files
parts = 4


for i_run in range(237, 247): 

    archive = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)
    outdir = OutputFolder

    split_lhco_auto_py2(archive, outdir, parts=parts)

    print('\n DONE file: ', archive)
    print('\n --------------------- \n')
    

('Total events found:', 617375)
('Headers saved to:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/bbaj_pc_MG5332_run_237_headers.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/bbaj_pc_MG5332_run_237_part00.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/bbaj_pc_MG5332_run_237_part01.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/bbaj_pc_MG5332_run_237_part02.lhco')
('Writing:', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/bbaj_pc_MG5332_run_237_part03.lhco')
DONE.
('\n DONE file: ', '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/bbaj_pc_MG5332_run_237.lhco.gz')

 --------------------- 

('Total events found:', 618006)
('Heade

In [72]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/'
FileName = 'bbaj_pc_MG5332_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/backgrounds/bbaj_wFAKES.h5", "a")


# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0
events_ok_weight_method = 0


for i_run in range(237, 247): 

    for i_part in range(0,4):

        file_path = Folder + FileName + "run_{:02d}".format(i_run) + "_part{:02d}.lhco".format(i_part)
    
        # If the file does not exist, skip and go to the next one
        if not os.path.isfile(file_path):
            print("[WARNING] File not found: {} → skipping".format(file_path))
            continue
            
        datarun = LHCO_reader.Events(f_name=file_path)
        
        # show file characteristics and number of generated events
        print(datarun)
        initial_evs += len(datarun)
    
    
        for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar
    
            #################
            # WEIGHT METHOD #
            #################

            weight_ev = weight_per_event(event, fake_rate_jet_to_photon, basic_id_cuts, selection_cuts)

            # count the number of expected events that passed everything with the weight method
            events_ok_weight_method += weight_ev


            
            ####################################
            # MISIDENTIFICATION (JET → PHOTON) #
            ####################################
            
            event = jet_to_photon_misID(event, fake_rate_jet_to_photon)

            
    
            #####################
            # BASIC PARTICLE ID #
            #####################
    
            index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)
    
    
            
            ############################
            # ESPECIFIC SELECTION CUTS #
            ############################
    
            passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)
    
            if not passed:
                continue # skip and go to the next event
                
    
    
            ##############
            # SAVE EVENT #
            ##############
    
            photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
            photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
            photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
            
            btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
            btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
            btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]
    
            jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
            jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
            jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]
    
            MET_pts  = [event["MET"][0]["PT"]]
            MET_phis = [event["MET"][0]["phi"]]
    
            # save the event
            N_saved = save_event(   FileSave, N_saved,
                                    photon_pts, photon_etas, photon_phis,
                                    btag_pts, btag_etas, btag_phis,
                                    jet_pts, jet_etas, jet_phis,
                                    MET_pts, MET_phis                       )
    
    
    
            # count the number of events that passed everything so far
            events_ok += 1
    
    
        pass # for the progress bar

        print('run: ',i_run, '   part:', i_part,  '   # events ok so far: ' , events_ok)
        

    # print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)
print('Total events after cuts (weights): ', events_ok_weight_method)

+------------------+---------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 154344                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/bbaj_pc_MG5332_run_237_part00.lhco |
+------------------+---------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 154344/154344 [00:03<00:00, 43402.06it/s]


('run: ', 237, '   part:', 0, '   # events ok so far: ', 11)
+------------------+---------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 154344                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/bbaj_pc_MG5332_run_237_part01.lhco |
+------------------+---------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 154344/154344 [00:03<00:00, 39683.64it/s]


('run: ', 237, '   part:', 1, '   # events ok so far: ', 22)
+------------------+---------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 154344                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/bbaj_pc_MG5332_run_237_part02.lhco |
+------------------+---------------------------------------------------------------------------------------------------------------------------------------+


...

('run: ', 246, '   part:', 1, '   # events ok so far: ', 460)
+------------------+---------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 154279                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/bbaj_pc_MG5332_run_246_part02.lhco |
+------------------+---------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 154279/154279 [00:04<00:00, 38499.56it/s]


('run: ', 246, '   part:', 2, '   # events ok so far: ', 468)
+------------------+---------------------------------------------------------------------------------------------------------------------------------------+
| Number of events | 154278                                                                                                                                |
| Description      | /home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/backgrounds/bbaj/DIVIDE_bbaj_pc/bbaj_pc_MG5332_run_246_part03.lhco |
+------------------+---------------------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 154278/154278 [00:04<00:00, 36160.07it/s]


('run: ', 246, '   part:', 3, '   # events ok so far: ', 476)
 
('Total initial events: ', 6177657)
('Total events after cuts: ', 476)
('Total events after cuts (weights): ', 58.050000000027175)


In [73]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)
print('Total events after cuts (weights): ', events_ok_weight_method)

('Total initial events: ', 6177657)
('Total events after cuts: ', 476)
('Total events after cuts (weights): ', 58.050000000027175)


In [33]:
initial_evs = 6177657
events_ok = 476
events_ok_weight_method = 58.050000000027175

In [74]:
cross_fb = MG_cross_bbaj*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")


aceptancia_W = 1.*events_ok_weight_method/initial_evs

fid_cross_W = cross_fb * aceptancia_W
Tot_ev_expected_W = cross_fb * aceptancia_W * luminosity

print('')
print('Fiducial cross section (weight method): ', fid_cross_W, "[fb]")
print('Events expected (weight method): ', Tot_ev_expected_W, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 20.64978643545603, '[fb]')
('Events expected: ', 61949.35930636809, '    for L=', 3000, ' [fb-1]')

('Fiducial cross section (weight method): ', 2.5183195432327388, '[fb]')
('Events expected (weight method): ', 7554.958629698216, '    for L=', 3000, ' [fb-1]')


##### this is the same as above (the last bbaj without fakes)

In [33]:
initial_evs = 6177657
events_ok = 412

In [34]:
cross_fb = MG_cross_bbaj*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 17.87334456178127, '[fb]')
('Events expected: ', 53620.03368534381, '    for L=', 3000, ' [fb-1]')


In [1]:
61949-7554

54395

# SM Higgs

## BP0 SM higgs double production

In [12]:
BPnum = 0

Folder = "../DATA/SMhiggs/"

file_path = Folder + "SMhiggs_run_03.lhco"
# file_path = Folder + "SMhiggs_run_04.lhco"
# file_path = Folder + "SMhiggs_run_07.lhco"
# file_path = Folder + "SMhiggs_run_12.lhco"


MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001770462175, '[pb]')


In [13]:

# File to save the processed data:
FileSave = h5py.File("../DATA/SMhiggs/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(3,13):

    file_path = Folder + "SMhiggs_run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:361: UserWarning: Did not parse all events in file
  warnings.warn("Did not parse all events in file")


+------------------+-------------------------------------+
| Number of events | 66659                               |
| Description      | ../DATA/SMhiggs/SMhiggs_run_03.lhco |
+------------------+-------------------------------------+


Processing events: 100%|██████████| 66659/66659 [00:15<00:00, 4171.36it/s]


('run: ', 3, '   # events ok so far: ', 4469)
+------------------+-------------------------------------+
| Number of events | 66823                               |
| Description      | ../DATA/SMhiggs/SMhiggs_run_04.lhco |
+------------------+-------------------------------------+


Processing events: 100%|██████████| 66823/66823 [00:29<00:00, 2275.14it/s]


...

('run: ', 10, '   # events ok so far: ', 36894)
+------------------+-------------------------------------+
| Number of events | 66664                               |
| Description      | ../DATA/SMhiggs/SMhiggs_run_11.lhco |
+------------------+-------------------------------------+


Processing events: 100%|██████████| 66664/66664 [00:32<00:00, 2077.97it/s]


('run: ', 11, '   # events ok so far: ', 41488)
+------------------+-------------------------------------+
| Number of events | 66661                               |
| Description      | ../DATA/SMhiggs/SMhiggs_run_12.lhco |
+------------------+-------------------------------------+


Processing events: 100%|██████████| 66661/66661 [00:29<00:00, 2223.09it/s]

('run: ', 12, '   # events ok so far: ', 46194)
 
('Total initial events: ', 667347)
('Total events after cuts: ', 46194)


In [14]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 667347)
('Total events after cuts: ', 46194)


In [15]:
initial_evs = 667347
events_ok = 46194

In [16]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.01225520302210844, '[fb]')
('Events expected: ', 36.76560906632532, '    for L=', 3000, ' [fb-1]')


# SIGNAL

## Point 1

#### files like: point-1_cluster_MG5332_ (.gz)

In [10]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/signal/point-1/'
file_path = Folder + "point-1_cluster_MG5332_run_23.lhco.gz"

MG_cross_point1 = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_point1, "[pb]")

('Cross section (MG) = ', 0.0003253938518125001, '[pb]')


In [11]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/signal/point-1/'
FileName = 'point-1_cluster_MG5332_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/point1.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(23, 32):  

    archive_path = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)

    if not os.path.isfile(archive_path):
        print("[WARNING] File not found: {} -> skipping".format(archive_path))
        continue

    # create a tmp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".lhco")
    tmp_path = tmp.name

    with gzip.open(archive_path, "rb") as gz, open(tmp_path, "wb") as out:
        out.write(gz.read())

    # read the LHCO
    datarun = LHCO_reader.Events(tmp_path)

   
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:361: UserWarning: Did not parse all events in file
  warnings.warn("Did not parse all events in file")


+------------------+---------------------+
| Number of events | 65492               |
| Description      | /tmp/tmp2JKw6X.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65492/65492 [00:18<00:00, 3628.71it/s]


('run: ', 23, '   # events ok so far: ', 5310)
+------------------+---------------------+
| Number of events | 65747               |
| Description      | /tmp/tmpDm5bh1.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65747/65747 [00:17<00:00, 3700.24it/s]


...

('run: ', 29, '   # events ok so far: ', 37077)
+------------------+---------------------+
| Number of events | 65677               |
| Description      | /tmp/tmpOuLf9H.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65677/65677 [00:20<00:00, 3184.48it/s]


('run: ', 30, '   # events ok so far: ', 42409)
+------------------+---------------------+
| Number of events | 65686               |
| Description      | /tmp/tmplpThgB.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65686/65686 [00:17<00:00, 3842.42it/s]

('run: ', 31, '   # events ok so far: ', 47769)
 
('Total initial events: ', 590539)
('Total events after cuts: ', 47769)


In [12]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 590539)
('Total events after cuts: ', 47769)


In [14]:
initial_evs = 590539
events_ok = 47769

In [13]:
cross_fb = MG_cross_point1*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.026321274136392882, '[fb]')
('Events expected: ', 78.96382240917865, '    for L=', 3000, ' [fb-1]')


#### files like: point-1_cluster_MG5332v2_ (.gz)

In [14]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/signal/point-1/'
file_path = Folder + "point-1_cluster_MG5332v2_run_14.lhco.gz"

MG_cross_point1 = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_point1, "[pb]")

('Cross section (MG) = ', 0.00032597913175, '[pb]')


In [15]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/signal/point-1/'
FileName = 'point-1_cluster_MG5332v2_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/point1.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(14, 20):  

    archive_path = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)

    if not os.path.isfile(archive_path):
        print("[WARNING] File not found: {} -> skipping".format(archive_path))
        continue

    # create a tmp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".lhco")
    tmp_path = tmp.name

    with gzip.open(archive_path, "rb") as gz, open(tmp_path, "wb") as out:
        out.write(gz.read())

    # read the LHCO
    datarun = LHCO_reader.Events(tmp_path)

   
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+---------------------+
| Number of events | 65544               |
| Description      | /tmp/tmpJuHArY.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65544/65544 [00:21<00:00, 3010.37it/s]


('run: ', 14, '   # events ok so far: ', 5326)
+------------------+---------------------+
| Number of events | 65530               |
| Description      | /tmp/tmp1AA25l.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65530/65530 [00:17<00:00, 3744.85it/s]


('run: ', 15, '   # events ok so far: ', 10771)
+------------------+---------------------+
| Number of events | 65530               |
| Description      | /tmp/tmprJx1rr.lhco |
+------------------+---------------------+


...

('run: ', 17, '   # events ok so far: ', 21350)
+------------------+---------------------+
| Number of events | 65490               |
| Description      | /tmp/tmpQRGXzE.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65490/65490 [00:18<00:00, 3513.19it/s]


('run: ', 18, '   # events ok so far: ', 26642)
+------------------+---------------------+
| Number of events | 65456               |
| Description      | /tmp/tmpRyiSDf.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65456/65456 [00:14<00:00, 4449.69it/s]

('run: ', 19, '   # events ok so far: ', 31722)
 
('Total initial events: ', 393275)
('Total events after cuts: ', 31722)


In [16]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 393275)
('Total events after cuts: ', 31722)


In [14]:
initial_evs = 393275
events_ok = 31722

In [17]:
cross_fb = MG_cross_point1*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.02629384023233997, '[fb]')
('Events expected: ', 78.88152069701991, '    for L=', 3000, ' [fb-1]')


#### files like: point-1_cluster_MG5332v4_ (.gz)  (runs 1 to 7)

In [22]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/signal/point-1/'
file_path = Folder + "point-1_cluster_MG5332v4_run_1.lhco.gz"

MG_cross_point1 = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_point1, "[pb]")

('Cross section (MG) = ', 0.000323614778, '[pb]')


In [25]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/signal/point-1/'
FileName = 'point-1_cluster_MG5332v4_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/point1.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(1, 8):  

    archive_path = Folder + FileName + "run_{:1d}.lhco.gz".format(i_run)

    if not os.path.isfile(archive_path):
        print("[WARNING] File not found: {} -> skipping".format(archive_path))
        continue

    # create a tmp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".lhco")
    tmp_path = tmp.name

    with gzip.open(archive_path, "rb") as gz, open(tmp_path, "wb") as out:
        out.write(gz.read())

    # read the LHCO
    datarun = LHCO_reader.Events(tmp_path)

   
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+---------------------+
| Number of events | 67563               |
| Description      | /tmp/tmp63UMfa.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 67563/67563 [00:17<00:00, 3919.11it/s]


('run: ', 1, '   # events ok so far: ', 5368)
+------------------+---------------------+
| Number of events | 67678               |
| Description      | /tmp/tmp_5j3r8.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 67678/67678 [00:16<00:00, 3996.25it/s]


...

('run: ', 5, '   # events ok so far: ', 26703)
+------------------+---------------------+
| Number of events | 67488               |
| Description      | /tmp/tmp44pZVk.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 67488/67488 [00:17<00:00, 3832.23it/s]


('run: ', 6, '   # events ok so far: ', 32005)
+------------------+---------------------+
| Number of events | 67583               |
| Description      | /tmp/tmpVkLcCQ.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 67583/67583 [00:19<00:00, 3514.98it/s]

('run: ', 7, '   # events ok so far: ', 37354)
 
('Total initial events: ', 473101)
('Total events after cuts: ', 37354)


In [26]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 473101)
('Total events after cuts: ', 37354)


In [14]:
initial_evs = 473101
events_ok = 37354

In [27]:
cross_fb = MG_cross_point1*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.025551217218758786, '[fb]')
('Events expected: ', 76.65365165627635, '    for L=', 3000, ' [fb-1]')


#### files like: point-1_cluster_MG5332v4_ (.gz)  (runs 22 to 36)

In [29]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/signal/point-1/'
file_path = Folder + "point-1_cluster_MG5332v4_run_22.lhco.gz"

MG_cross_point1 = get_cross_section_from_lhco_gz(file_path)

print("Cross section (MG) = ", MG_cross_point1, "[pb]")

('Cross section (MG) = ', 0.00032627816562500004, '[pb]')


In [30]:
Folder = '/home/andres/CompuTools/Programas/Other_MG5_data/Higgs_double_prod/signal/point-1/'
FileName = 'point-1_cluster_MG5332v4_'   

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/point1.h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this FileName
initial_evs = 0
events_ok = 0


for i_run in range(22, 36):  

    archive_path = Folder + FileName + "run_{:02d}.lhco.gz".format(i_run)

    if not os.path.isfile(archive_path):
        print("[WARNING] File not found: {} -> skipping".format(archive_path))
        continue

    # create a tmp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".lhco")
    tmp_path = tmp.name

    with gzip.open(archive_path, "rb") as gz, open(tmp_path, "wb") as out:
        out.write(gz.read())

    # read the LHCO
    datarun = LHCO_reader.Events(tmp_path)

   
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+---------------------+
| Number of events | 65634               |
| Description      | /tmp/tmpHjGyPs.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65634/65634 [00:16<00:00, 4059.32it/s]


('run: ', 22, '   # events ok so far: ', 5401)
+------------------+---------------------+
| Number of events | 65485               |
| Description      | /tmp/tmp2_i6WV.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65485/65485 [00:16<00:00, 4016.44it/s]


('run: ', 23, '   # events ok so far: ', 10822)
+------------------+---------------------+
| Number of events | 65655               |
| Description      | /tmp/tmpIyLpRr.lhco |
+------------------+---------------------+


...

('run: ', 33, '   # events ok so far: ', 64147)
+------------------+---------------------+
| Number of events | 65523               |
| Description      | /tmp/tmpsX_5dF.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65523/65523 [00:17<00:00, 3642.73it/s]


('run: ', 34, '   # events ok so far: ', 69498)
+------------------+---------------------+
| Number of events | 65668               |
| Description      | /tmp/tmp3Nu4Gt.lhco |
+------------------+---------------------+


Processing events: 100%|██████████| 65668/65668 [00:20<00:00, 3129.49it/s]

('run: ', 35, '   # events ok so far: ', 74832)
 
('Total initial events: ', 918167)
('Total events after cuts: ', 74832)


In [31]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 918167)
('Total events after cuts: ', 74832)


In [14]:
initial_evs = 918167
events_ok = 74832

In [32]:
cross_fb = MG_cross_point1*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.02659216426864612, '[fb]')
('Events expected: ', 79.77649280593836, '    for L=', 3000, ' [fb-1]')


In [23]:
0.00032627816562500004*1000

0.32627816562500006

In [24]:
0.32*3000

960.0

In [25]:
960*(74832./918167)

78.24145280760472

In [26]:
0.018950501462499998*1000

18.950501462499997

In [27]:
18.95*3000

56850.0

In [28]:
56850*(74832./918167)

4633.361033450342

# CLUSTER Andres

## BP28 (done)

In [25]:
BPnum = 28

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
# file_path = Folder + "run_01/run_01_tag_2_banner.txt"
file_path = Folder + "run_02/run_02_tag_2_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0003116061476250001, '[pb]')


In [17]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,3):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+---------------------------------------------------------------------------------------------------+
| Number of events | 65347                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP28/lhco/run_01.lhco |
+------------------+---------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65347/65347 [00:26<00:00, 2495.69it/s]


('run: ', 1, '   # events ok so far: ', 5261)
+------------------+---------------------------------------------------------------------------------------------------+
| Number of events | 47291                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP28/lhco/run_02.lhco |
+------------------+---------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 47291/47291 [00:12<00:00, 3683.22it/s]

('run: ', 2, '   # events ok so far: ', 9033)
 
('Total initial events: ', 112638)
('Total events after cuts: ', 9033)


In [18]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 112638)
('Total events after cuts: ', 9033)


In [19]:
initial_evs = 112638
events_ok = 9033

In [20]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.024989242808791227, '[fb]')
('Events expected: ', 74.96772842637368, '    for L=', 3000, ' [fb-1]')


## BP62 (done)

In [27]:
BPnum = 62

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
# file_path = Folder + "run_01/run_01_tag_6_banner.txt"
file_path = Folder + "run_02/run_02_tag_5_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000256581439875, '[pb]')


In [28]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,3):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+---------------------------------------------------------------------------------------------------+
| Number of events | 65257                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP62/lhco/run_01.lhco |
+------------------+---------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65257/65257 [00:16<00:00, 4067.42it/s]


('run: ', 1, '   # events ok so far: ', 4966)
+------------------+---------------------------------------------------------------------------------------------------+
| Number of events | 65273                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP62/lhco/run_02.lhco |
+------------------+---------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65273/65273 [00:32<00:00, 1992.78it/s]

('run: ', 2, '   # events ok so far: ', 9919)
 
('Total initial events: ', 130530)
('Total events after cuts: ', 9919)


In [29]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130530)
('Total events after cuts: ', 9919)


In [30]:
initial_evs = 130530
events_ok = 9919

In [31]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.019497673348043552, '[fb]')
('Events expected: ', 58.49302004413066, '    for L=', 3000, ' [fb-1]')


## BP63 (done)

In [33]:
BPnum = 63

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
# file_path = Folder + "run_01/run_01_tag_3_banner.txt"
file_path = Folder + "run_02/run_02_tag_1_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0002985677431875, '[pb]')


In [34]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,3):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+---------------------------------------------------------------------------------------------------+
| Number of events | 65346                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP63/lhco/run_01.lhco |
+------------------+---------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65346/65346 [00:24<00:00, 2642.29it/s]


('run: ', 1, '   # events ok so far: ', 5077)
+------------------+---------------------------------------------------------------------------------------------------+
| Number of events | 65584                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP63/lhco/run_02.lhco |
+------------------+---------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65584/65584 [00:32<00:00, 1989.51it/s]

('run: ', 2, '   # events ok so far: ', 10295)
 
('Total initial events: ', 130930)
('Total events after cuts: ', 10295)


In [35]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130930)
('Total events after cuts: ', 10295)


In [36]:
initial_evs = 130930
events_ok = 10295

In [37]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.023476322585467904, '[fb]')
('Events expected: ', 70.42896775640371, '    for L=', 3000, ' [fb-1]')


## BP219 (done)

In [39]:
BPnum = 219

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
# file_path = Folder + "run_01/run_01_tag_4_banner.txt"
file_path = Folder + "run_02/run_02_tag_3_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0002861830652499999, '[pb]')


In [40]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,3):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 65372                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP219/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65372/65372 [00:35<00:00, 1853.99it/s]


('run: ', 1, '   # events ok so far: ', 4916)
+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 65529                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP219/lhco/run_02.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65529/65529 [00:32<00:00, 2040.91it/s]

('run: ', 2, '   # events ok so far: ', 9757)
 
('Total initial events: ', 130901)
('Total events after cuts: ', 9757)


In [41]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130901)
('Total events after cuts: ', 9757)


In [42]:
initial_evs = 130901
events_ok = 9757

In [43]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.021331297451083257, '[fb]')
('Events expected: ', 63.99389235324977, '    for L=', 3000, ' [fb-1]')


## BP220 (done)

In [48]:
BPnum = 220

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
# file_path = Folder + "run_01/run_01_tag_5_banner.txt"
# file_path = Folder + "run_02/run_02_tag_4_banner.txt"
file_path = Folder + "run_03/run_03_tag_5_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0002636812626875, '[pb]')


In [49]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,4):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 65604                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP220/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65604/65604 [00:34<00:00, 1901.98it/s]


('run: ', 1, '   # events ok so far: ', 4766)
+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 65358                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP220/lhco/run_02.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65358/65358 [00:15<00:00, 4155.72it/s]


('run: ', 2, '   # events ok so far: ', 9626)
+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 65150                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP220/lhco/run_03.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65150/65150 [00:17<00:00, 3690.54it/s]

('run: ', 3, '   # events ok so far: ', 14460)
 
('Total initial events: ', 196112)
('Total events after cuts: ', 14460)


In [50]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 196112)
('Total events after cuts: ', 14460)


In [51]:
initial_evs = 196112
events_ok = 14460

In [52]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.019442109908935964, '[fb]')
('Events expected: ', 58.32632972680789, '    for L=', 3000, ' [fb-1]')


## BP626 (done)

In [20]:
BPnum = 626

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00025206004387499996, '[pb]')


In [21]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130818                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP626/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130818/130818 [00:36<00:00, 3548.61it/s]

('run: ', 1, '   # events ok so far: ', 9982)
 
('Total initial events: ', 130818)
('Total events after cuts: ', 9982)


In [22]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130818)
('Total events after cuts: ', 9982)


In [23]:
initial_evs = 130818
events_ok = 9982

In [24]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.019233311608190386, '[fb]')
('Events expected: ', 57.69993482457116, '    for L=', 3000, ' [fb-1]')


## BP658 (done)

In [20]:
BPnum = 658

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00020199046650000002, '[pb]')


In [21]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130437                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP658/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130437/130437 [00:31<00:00, 4174.22it/s]

('run: ', 1, '   # events ok so far: ', 9030)
 
('Total initial events: ', 130437)
('Total events after cuts: ', 9030)


In [22]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130437)
('Total events after cuts: ', 9030)


In [23]:
initial_evs = 130437
events_ok = 9030

In [24]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.013983562275236321, '[fb]')
('Events expected: ', 41.950686825708964, '    for L=', 3000, ' [fb-1]')


## BP668 (done)

In [36]:
BPnum = 668

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00020328925125, '[pb]')


In [37]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130385                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP668/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130385/130385 [00:33<00:00, 3938.46it/s]

('run: ', 1, '   # events ok so far: ', 9052)
 
('Total initial events: ', 130385)
('Total events after cuts: ', 9052)


In [38]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130385)
('Total events after cuts: ', 9052)


In [39]:
initial_evs = 130385
events_ok = 9052

In [40]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.014113389594777006, '[fb]')
('Events expected: ', 42.34016878433102, '    for L=', 3000, ' [fb-1]')


## BP673 (done)

In [30]:
BPnum = 673

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_7_banner.txt"
file_path = Folder + "run_02/run_01_tag_22_banner.txt"
# file_path = Folder + "run_03/run_03_tag_5_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0002304566725, '[pb]')


In [32]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,3):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 65075                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP673/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65075/65075 [00:16<00:00, 4051.87it/s]


('run: ', 1, '   # events ok so far: ', 4809)
+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130857                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP673/lhco/run_02.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130857/130857 [00:30<00:00, 4285.68it/s]

('run: ', 2, '   # events ok so far: ', 14431)
 
('Total initial events: ', 195932)
('Total events after cuts: ', 14431)


In [33]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 195932)
('Total events after cuts: ', 14431)


In [34]:
initial_evs = 195932
events_ok = 14431

In [35]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.016973849298978726, '[fb]')
('Events expected: ', 50.92154789693618, '    for L=', 3000, ' [fb-1]')


## BP678 (done)

In [8]:
BPnum = 678

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00022835563987499998, '[pb]')


In [9]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130513                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP678/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130513/130513 [00:30<00:00, 4296.79it/s]

('run: ', 1, '   # events ok so far: ', 9163)
 
('Total initial events: ', 130513)
('Total events after cuts: ', 9163)


In [10]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130513)
('Total events after cuts: ', 9163)


In [11]:
initial_evs = 130513
events_ok = 9163

In [12]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.016032293550639592, '[fb]')
('Events expected: ', 48.09688065191878, '    for L=', 3000, ' [fb-1]')


## BP684 (done)

In [14]:
BPnum = 684

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_8_banner.txt"
file_path = Folder + "run_02/run_01_tag_22_banner.txt"
# file_path = Folder + "run_03/run_03_tag_5_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00023024137437499997, '[pb]')


In [15]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,3):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 65488                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP684/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65488/65488 [00:18<00:00, 3630.22it/s]


('run: ', 1, '   # events ok so far: ', 4747)
+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130532                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP684/lhco/run_02.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130532/130532 [00:32<00:00, 4069.27it/s]

('run: ', 2, '   # events ok so far: ', 14448)
 
('Total initial events: ', 196020)
('Total events after cuts: ', 14448)


In [16]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 196020)
('Total events after cuts: ', 14448)


In [17]:
initial_evs = 196020
events_ok = 14448

In [18]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.016970346785889195, '[fb]')
('Events expected: ', 50.91104035766758, '    for L=', 3000, ' [fb-1]')


## BP685 (done)

In [19]:
BPnum = 685

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_9_banner.txt"
file_path = Folder + "run_02/run_01_tag_22_banner.txt"
# file_path = Folder + "run_03/run_03_tag_5_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0002020995569375, '[pb]')


In [20]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,3):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 65257                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP685/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65257/65257 [00:14<00:00, 4429.94it/s]


('run: ', 1, '   # events ok so far: ', 4542)
+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130209                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP685/lhco/run_02.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130209/130209 [00:29<00:00, 4470.42it/s]

('run: ', 2, '   # events ok so far: ', 13637)
 
('Total initial events: ', 195466)
('Total events after cuts: ', 13637)


In [21]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 195466)
('Total events after cuts: ', 13637)


In [22]:
initial_evs = 195466
events_ok = 13637

In [23]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.014099800773314474, '[fb]')
('Events expected: ', 42.299402319943425, '    for L=', 3000, ' [fb-1]')


## BP728 (done)

In [14]:
BPnum = 728

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
# file_path = Folder + "run_01/run_01_tag_10_banner.txt"
file_path = Folder + "run_02/run_01_tag_22_banner.txt"
# file_path = Folder + "run_03/run_03_tag_5_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001749166541875, '[pb]')


In [15]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,3):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 64893                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP728/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 64893/64893 [00:15<00:00, 4280.63it/s]


('run: ', 1, '   # events ok so far: ', 4451)
+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130424                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP728/lhco/run_02.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130424/130424 [00:56<00:00, 2298.78it/s]

('run: ', 2, '   # events ok so far: ', 13455)
 
('Total initial events: ', 195317)
('Total events after cuts: ', 13455)


In [16]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 195317)
('Total events after cuts: ', 13455)


In [17]:
initial_evs = 195317
events_ok = 13455

In [18]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.012049660716132301, '[fb]')
('Events expected: ', 36.148982148396904, '    for L=', 3000, ' [fb-1]')


## BP732 (done)

In [11]:
BPnum = 732

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_17_banner.txt"
# file_path = Folder + "run_02/run_02_tag_4_banner.txt"
# file_path = Folder + "run_03/run_03_tag_5_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00017536720718750002, '[pb]')


In [12]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130341                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP732/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130341/130341 [00:31<00:00, 4135.39it/s]

('run: ', 1, '   # events ok so far: ', 9027)
 
('Total initial events: ', 130341)
('Total events after cuts: ', 9027)


In [13]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130341)
('Total events after cuts: ', 9027)


In [14]:
initial_evs = 130341
events_ok = 9027

In [15]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.012145370829451689, '[fb]')
('Events expected: ', 36.436112488355064, '    for L=', 3000, ' [fb-1]')


## BP752 (done)

In [34]:
BPnum = 752

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"
# file_path = Folder + "run_02/run_02_tag_4_banner.txt"
# file_path = Folder + "run_03/run_03_tag_5_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001741906340625, '[pb]')


In [35]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130346                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP752/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130346/130346 [00:27<00:00, 4783.96it/s]

('run: ', 1, '   # events ok so far: ', 8958)
 
('Total initial events: ', 130346)
('Total events after cuts: ', 8958)


In [36]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130346)
('Total events after cuts: ', 8958)


In [37]:
initial_evs = 130346
events_ok = 8958

In [38]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.011971212771637601, '[fb]')
('Events expected: ', 35.9136383149128, '    for L=', 3000, ' [fb-1]')


## BP754 (done)

In [9]:
BPnum = 754

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_11_banner.txt"
# file_path = Folder + "run_02/run_02_tag_4_banner.txt"
# file_path = Folder + "run_03/run_03_tag_5_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000201750694625, '[pb]')


In [10]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,3):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 65257                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP754/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65257/65257 [00:15<00:00, 4087.81it/s]


('run: ', 1, '   # events ok so far: ', 4702)
+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 65152                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP754/lhco/run_02.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65152/65152 [00:37<00:00, 1760.44it/s]

('run: ', 2, '   # events ok so far: ', 9477)
 
('Total initial events: ', 130409)
('Total events after cuts: ', 9477)


In [11]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130409)
('Total events after cuts: ', 9477)


In [12]:
initial_evs = 130409
events_ok = 9477

In [13]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.014661498308867675, '[fb]')
('Events expected: ', 43.98449492660303, '    for L=', 3000, ' [fb-1]')


## BP758 (done)

In [14]:
BPnum = 758

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_12_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00020481644112500002, '[pb]')


In [15]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130607                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP758/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130607/130607 [00:45<00:00, 2857.85it/s]

('run: ', 1, '   # events ok so far: ', 9572)
 
('Total initial events: ', 130607)
('Total events after cuts: ', 9572)


In [16]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130607)
('Total events after cuts: ', 9572)


In [17]:
initial_evs = 130607
events_ok = 9572

In [18]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.015010703671690648, '[fb]')
('Events expected: ', 45.03211101507194, '    for L=', 3000, ' [fb-1]')


## BP763 (done)

In [8]:
BPnum = 763

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_13_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00017467851625, '[pb]')


In [9]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 129989                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP763/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129989/129989 [00:40<00:00, 3206.81it/s]

('run: ', 1, '   # events ok so far: ', 8900)
 
('Total initial events: ', 129989)
('Total events after cuts: ', 8900)


In [10]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129989)
('Total events after cuts: ', 8900)


In [11]:
initial_evs = 129989
events_ok = 8900

In [12]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.011959771939356408, '[fb]')
('Events expected: ', 35.87931581806922, '    for L=', 3000, ' [fb-1]')


## BP765 (done)

In [8]:
BPnum = 765

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_14_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001984628640625, '[pb]')


In [9]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130527                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP765/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130527/130527 [00:29<00:00, 4424.57it/s]

('run: ', 1, '   # events ok so far: ', 9134)
 
('Total initial events: ', 130527)
('Total events after cuts: ', 9134)


In [10]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130527)
('Total events after cuts: ', 9134)


In [11]:
initial_evs = 130527
events_ok = 9134

In [12]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.01388800631552763, '[fb]')
('Events expected: ', 41.66401894658289, '    for L=', 3000, ' [fb-1]')


## BP787 (done)

In [8]:
BPnum = 787

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_15_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00022209302781249997, '[pb]')


In [9]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130656                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP787/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130656/130656 [00:39<00:00, 3297.09it/s]

('run: ', 1, '   # events ok so far: ', 9869)
 
('Total initial events: ', 130656)
('Total events after cuts: ', 9869)


In [10]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130656)
('Total events after cuts: ', 9869)


In [11]:
initial_evs = 130656
events_ok = 9869

In [12]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.01677562524094999, '[fb]')
('Events expected: ', 50.32687572284998, '    for L=', 3000, ' [fb-1]')


## BP791 (done)

In [8]:
BPnum = 791

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000170050425375, '[pb]')


In [9]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130120                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP791/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130120/130120 [00:32<00:00, 4021.75it/s]

('run: ', 1, '   # events ok so far: ', 8786)
 
('Total initial events: ', 130120)
('Total events after cuts: ', 8786)


In [10]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130120)
('Total events after cuts: ', 8786)


In [11]:
initial_evs = 130120
events_ok = 8786

In [12]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.011482193646977789, '[fb]')
('Events expected: ', 34.446580940933366, '    for L=', 3000, ' [fb-1]')


## BP804 (NOT done)

In [28]:
BPnum = 804

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP826 (done)

In [13]:
BPnum = 826

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_20_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000174279917625, '[pb]')


In [14]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130057                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP826/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130057/130057 [00:29<00:00, 4391.30it/s]

('run: ', 1, '   # events ok so far: ', 9273)
 
('Total initial events: ', 130057)
('Total events after cuts: ', 9273)


In [15]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130057)
('Total events after cuts: ', 9273)


In [16]:
initial_evs = 130057
events_ok = 9273

In [17]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.012426072230918942, '[fb]')
('Events expected: ', 37.27821669275683, '    for L=', 3000, ' [fb-1]')


## BP841 (done)

In [18]:
BPnum = 841

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_21_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001655250511875, '[pb]')


In [19]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 127275                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP841/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 127275/127275 [00:32<00:00, 3885.20it/s]

('run: ', 1, '   # events ok so far: ', 9034)
 
('Total initial events: ', 127275)
('Total events after cuts: ', 9034)


In [20]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 127275)
('Total events after cuts: ', 9034)


In [21]:
initial_evs = 127275
events_ok = 9034

In [22]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.011748994794169123, '[fb]')
('Events expected: ', 35.24698438250737, '    for L=', 3000, ' [fb-1]')


## BP850 (done)

In [18]:
BPnum = 850

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_16_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000203983545125, '[pb]')


In [19]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130466                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP850/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130466/130466 [00:40<00:00, 3207.19it/s]

('run: ', 1, '   # events ok so far: ', 9421)
 
('Total initial events: ', 130466)
('Total events after cuts: ', 9421)


In [20]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130466)
('Total events after cuts: ', 9421)


In [21]:
initial_evs = 130466
events_ok = 9421

In [22]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.01472973018734862, '[fb]')
('Events expected: ', 44.18919056204586, '    for L=', 3000, ' [fb-1]')


## BP855 (done)

In [23]:
BPnum = 855

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_18_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00014175029537500002, '[pb]')


In [24]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130049                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP855/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130049/130049 [00:27<00:00, 4679.92it/s]

('run: ', 1, '   # events ok so far: ', 9312)
 
('Total initial events: ', 130049)
('Total events after cuts: ', 9312)


In [25]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130049)
('Total events after cuts: ', 9312)


In [26]:
initial_evs = 130049
events_ok = 9312

In [27]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010149856981076363, '[fb]')
('Events expected: ', 30.449570943229087, '    for L=', 3000, ' [fb-1]')


## BP872 (done)

In [8]:
BPnum = 872

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_19_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001474474305, '[pb]')


In [9]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 129832                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP872/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129832/129832 [00:33<00:00, 3873.45it/s]

('run: ', 1, '   # events ok so far: ', 9126)
 
('Total initial events: ', 129832)
('Total events after cuts: ', 9126)


In [10]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129832)
('Total events after cuts: ', 9126)


In [11]:
initial_evs = 129832
events_ok = 9126

In [12]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.01036420336082784, '[fb]')
('Events expected: ', 31.09261008248352, '    for L=', 3000, ' [fb-1]')


## BP875 (done)

In [24]:
BPnum = 875

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00014706520706249997, '[pb]')


In [25]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 129874                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP875/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129874/129874 [00:29<00:00, 4474.13it/s]

('run: ', 1, '   # events ok so far: ', 9224)
 
('Total initial events: ', 129874)
('Total events after cuts: ', 9224)


In [26]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129874)
('Total events after cuts: ', 9224)


In [27]:
initial_evs = 129874
events_ok = 9224

In [28]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010444965658596024, '[fb]')
('Events expected: ', 31.33489697578807, '    for L=', 3000, ' [fb-1]')


## BP881 (done)

In [29]:
BPnum = 881

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001476435029375, '[pb]')


In [30]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130131                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP881/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130131/130131 [00:44<00:00, 2948.40it/s]

('run: ', 1, '   # events ok so far: ', 9217)
 
('Total initial events: ', 130131)
('Total events after cuts: ', 9217)


In [31]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130131)
('Total events after cuts: ', 9217)


In [32]:
initial_evs = 130131
events_ok = 9217

In [33]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010457386530303599, '[fb]')
('Events expected: ', 31.372159590910798, '    for L=', 3000, ' [fb-1]')


## BP933 (NOT done)

In [29]:
BPnum = 933

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001476435029375, '[pb]')


In [30]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 130131                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP881/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130131/130131 [00:44<00:00, 2948.40it/s]

('run: ', 1, '   # events ok so far: ', 9217)
 
('Total initial events: ', 130131)
('Total events after cuts: ', 9217)


In [31]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130131)
('Total events after cuts: ', 9217)


In [32]:
initial_evs = 
events_ok = 

In [33]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010457386530303599, '[fb]')
('Events expected: ', 31.372159590910798, '    for L=', 3000, ' [fb-1]')


## BP996 (done)

In [41]:
BPnum = 996

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001468959215, '[pb]')


In [42]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+----------------------------------------------------------------------------------------------------+
| Number of events | 129644                                                                                             |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP996/lhco/run_01.lhco |
+------------------+----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129644/129644 [00:32<00:00, 4025.00it/s]

('run: ', 1, '   # events ok so far: ', 9123)
 
('Total initial events: ', 129644)
('Total events after cuts: ', 9123)


In [43]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129644)
('Total events after cuts: ', 9123)


In [44]:
initial_evs = 129644
events_ok = 9123

In [45]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010337011291262996, '[fb]')
('Events expected: ', 31.01103387378899, '    for L=', 3000, ' [fb-1]')


## BP1017 (done)

In [8]:
BPnum = 1017

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001461926333125, '[pb]')


In [9]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129747                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1017/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129747/129747 [00:27<00:00, 4745.70it/s]

('run: ', 1, '   # events ok so far: ', 9054)
 
('Total initial events: ', 129747)
('Total events after cuts: ', 9054)


In [10]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129747)
('Total events after cuts: ', 9054)


In [11]:
initial_evs = 129747
events_ok = 9054

In [12]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010201608530535389, '[fb]')
('Events expected: ', 30.60482559160617, '    for L=', 3000, ' [fb-1]')


## BP1035 (done)

In [13]:
BPnum = 1035

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001453263253125, '[pb]')


In [14]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129462                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1035/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129462/129462 [00:47<00:00, 2730.56it/s]

('run: ', 1, '   # events ok so far: ', 9077)
 
('Total initial events: ', 129462)
('Total events after cuts: ', 9077)


In [15]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129462)
('Total events after cuts: ', 9077)


In [16]:
initial_evs = 129462
events_ok = 9077

In [17]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.0101892992141444, '[fb]')
('Events expected: ', 30.567897642433202, '    for L=', 3000, ' [fb-1]')


## BP1036 (done)

In [13]:
BPnum = 1036

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00014596149225, '[pb]')


In [14]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129561                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1036/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129561/129561 [00:40<00:00, 3206.98it/s]

('run: ', 1, '   # events ok so far: ', 9264)
 
('Total initial events: ', 129561)
('Total events after cuts: ', 9264)


In [15]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129561)
('Total events after cuts: ', 9264)


In [16]:
initial_evs = 129561
events_ok = 9264

In [17]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010436684374186676, '[fb]')
('Events expected: ', 31.310053122560028, '    for L=', 3000, ' [fb-1]')


## BP1087 (done)

In [18]:
BPnum = 1087

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00014629361837499997, '[pb]')


In [19]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 130153                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1087/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 130153/130153 [00:37<00:00, 3450.94it/s]

('run: ', 1, '   # events ok so far: ', 9201)
 
('Total initial events: ', 130153)
('Total events after cuts: ', 9201)


In [20]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130153)
('Total events after cuts: ', 9201)


In [21]:
initial_evs = 130153
events_ok = 9201

In [22]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.01034204038837656, '[fb]')
('Events expected: ', 31.02612116512968, '    for L=', 3000, ' [fb-1]')


## BP1103 (done)

In [23]:
BPnum = 1103

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.0001470026726875, '[pb]')


In [24]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129808                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1103/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129808/129808 [00:38<00:00, 3411.36it/s]

('run: ', 1, '   # events ok so far: ', 9316)
 
('Total initial events: ', 129808)
('Total events after cuts: ', 9316)


In [25]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129808)
('Total events after cuts: ', 9316)


In [26]:
initial_evs = 129808
events_ok = 9316

In [27]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010550019249636002, '[fb]')
('Events expected: ', 31.650057748908004, '    for L=', 3000, ' [fb-1]')


## BP1104 (done)

In [28]:
BPnum = 1104

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 129817
events_ok = 9160

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP2381 (done)

In [46]:
BPnum = 2381

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015952686900000003, '[pb]')


In [47]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129749                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP2381/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129749/129749 [00:31<00:00, 4081.67it/s]

('run: ', 1, '   # events ok so far: ', 8930)
 
('Total initial events: ', 129749)
('Total events after cuts: ', 8930)


In [48]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129749)
('Total events after cuts: ', 8930)


In [49]:
initial_evs = 129749
events_ok = 8930

In [50]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.01097946758872901, '[fb]')
('Events expected: ', 32.93840276618703, '    for L=', 3000, ' [fb-1]')


## BP7307 (not done)

In [28]:
BPnum = 7307

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP9917 (not done)

In [28]:
BPnum = 9917

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP12218 (not done)

In [28]:
BPnum = 12218

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP14562 (not done)

In [28]:
BPnum = 14562

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP18630 (not done)

In [28]:
BPnum = 18630

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP27359 (not done)

In [28]:
BPnum = 27359

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP35018 (done)

In [8]:
BPnum = 35018

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00024761963075, '[pb]')


In [9]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:359: UserWarning: Couldn't read total number of events from LHCO
  warnings.warn("Couldn't read total number of events from LHCO")


+------------------+------------------------------------------------------------------------------------------------------+
| Number of events | 131016                                                                                               |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP35018/lhco/run_01.lhco |
+------------------+------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 131016/131016 [03:23<00:00, 642.36it/s] 

('run: ', 1, '   # events ok so far: ', 9356)
 
('Total initial events: ', 131016)
('Total events after cuts: ', 9356)


In [10]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 131016)
('Total events after cuts: ', 9356)


In [11]:
initial_evs = 131016
events_ok = 9356

In [12]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.017682796492771876, '[fb]')
('Events expected: ', 53.04838947831563, '    for L=', 3000, ' [fb-1]')


## BP35323 (not done)

In [28]:
BPnum = 35323

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP35390 (not done)

In [28]:
BPnum = 35390

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP35574 (not done)

In [28]:
BPnum = 35574

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


## BP35758 (not done)

In [28]:
BPnum = 35758

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP' + str(BPnum) + '/'
file_path = Folder + "run_01/run_01_tag_22_banner.txt"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00015009827743750001, '[pb]')


In [29]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,2):

    file_path = Folder + "lhco/run_{:02d}.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+-----------------------------------------------------------------------------------------------------+
| Number of events | 129817                                                                                              |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/ClusterAndres/BP1104/lhco/run_01.lhco |
+------------------+-----------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 129817/129817 [00:29<00:00, 4466.02it/s]

('run: ', 1, '   # events ok so far: ', 9160)
 
('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [30]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 129817)
('Total events after cuts: ', 9160)


In [31]:
initial_evs = 
events_ok = 

In [32]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.010591064508712266, '[fb]')
('Events expected: ', 31.773193526136797, '    for L=', 3000, ' [fb-1]')


# CLUSTER IB-LdR

## BP586 (done)

In [58]:
BPnum = 586

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/Leandro/BP' + str(BPnum) + '/'
# file_path = Folder + "run_01/tag_1_delphes_events.lhco"
# file_path = Folder + "run_02/tag_1_delphes_events.lhco"
file_path = Folder + "run_03/tag_1_delphes_events.lhco"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000257617686, '[pb]')


In [59]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,4):

    file_path = Folder + "run_{:02d}/tag_1_delphes_events.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

/home/andres/anaconda3/envs/py2/lib/python2.7/site-packages/LHCO_reader/LHCO_reader.py:361: UserWarning: Did not parse all events in file
  warnings.warn("Did not parse all events in file")


+------------------+--------------------------------------------------------------------------------------------------------------+
| Number of events | 65530                                                                                                        |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/Leandro/BP586/run_01/tag_1_delphes_events.lhco |
+------------------+--------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65530/65530 [00:14<00:00, 4546.70it/s]


('run: ', 1, '   # events ok so far: ', 4749)
+------------------+--------------------------------------------------------------------------------------------------------------+
| Number of events | 65365                                                                                                        |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/Leandro/BP586/run_02/tag_1_delphes_events.lhco |
+------------------+--------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65365/65365 [00:14<00:00, 4363.30it/s]


('run: ', 2, '   # events ok so far: ', 9667)
+------------------+--------------------------------------------------------------------------------------------------------------+
| Number of events | 65450                                                                                                        |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/Leandro/BP586/run_03/tag_1_delphes_events.lhco |
+------------------+--------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65450/65450 [00:14<00:00, 4483.66it/s]

('run: ', 3, '   # events ok so far: ', 14546)
 
('Total initial events: ', 196345)
('Total events after cuts: ', 14546)


In [50]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 196112)
('Total events after cuts: ', 14460)


In [60]:
initial_evs = 196112
events_ok = 14460

In [61]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.018995021924002615, '[fb]')
('Events expected: ', 56.98506577200784, '    for L=', 3000, ' [fb-1]')


## BP635 (done)

In [64]:
BPnum = 635

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/Leandro/BP' + str(BPnum) + '/'
# file_path = Folder + "run_01/tag_1_delphes_events.lhco"
file_path = Folder + "run_02/tag_1_delphes_events.lhco"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000230046225, '[pb]')


In [65]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(1,3):

    file_path = Folder + "run_{:02d}/tag_1_delphes_events.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+--------------------------------------------------------------------------------------------------------------+
| Number of events | 65324                                                                                                        |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/Leandro/BP635/run_01/tag_1_delphes_events.lhco |
+------------------+--------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65324/65324 [00:13<00:00, 4785.19it/s]


('run: ', 1, '   # events ok so far: ', 4767)
+------------------+--------------------------------------------------------------------------------------------------------------+
| Number of events | 65302                                                                                                        |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/Leandro/BP635/run_02/tag_1_delphes_events.lhco |
+------------------+--------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 65302/65302 [00:13<00:00, 4742.18it/s]

('run: ', 2, '   # events ok so far: ', 9597)
 
('Total initial events: ', 130626)
('Total events after cuts: ', 9597)


In [67]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130626)
('Total events after cuts: ', 9597)


In [68]:
initial_evs = 130626
events_ok = 9597

In [69]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.016901333741559874, '[fb]')
('Events expected: ', 50.70400122467962, '    for L=', 3000, ' [fb-1]')


# La Plata Team

## REDONE (SAME MODEL)

## BP502

In [14]:
BPnum = 502

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP' + str(BPnum) + '/'
file_path = Folder + "output_ggHH_BP502_69_0_delphes_events.lhco"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000289936176, '[pb]')


In [15]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(0,11):

    file_path = Folder + "output_ggHH_BP502_69_{:1d}_delphes_events.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13072                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP502/output_ggHH_BP502_69_0_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13072/13072 [00:03<00:00, 3701.19it/s]


('run: ', 0, '   # events ok so far: ', 976)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13056                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP502/output_ggHH_BP502_69_1_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13056/13056 [00:04<00:00, 2877.69it/s]


('run: ', 1, '   # events ok so far: ', 2007)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13154                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP502/output_ggHH_BP502_69_2_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


...

('run: ', 8, '   # events ok so far: ', 9082)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13116                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP502/output_ggHH_BP502_69_9_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13116/13116 [00:07<00:00, 1652.80it/s]


('run: ', 9, '   # events ok so far: ', 10108)
+------------------+-------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13101                                                                                                                   |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP502/output_ggHH_BP502_69_10_delphes_events.lhco |
+------------------+-------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13101/13101 [00:07<00:00, 1700.72it/s]

('run: ', 10, '   # events ok so far: ', 11160)
 
('Total initial events: ', 143966)
('Total events after cuts: ', 11160)


In [17]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 143966)
('Total events after cuts: ', 11160)


In [18]:
initial_evs = 143966
events_ok = 11160

In [19]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.022475360322298323, '[fb]')
('Events expected: ', 67.42608096689497, '    for L=', 3000, ' [fb-1]')


## WRONG MODEL

## BP279 (done)

In [41]:
BPnum = 279

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP' + str(BPnum) + '/'
file_path = Folder + "output_ggHH_BP279_42_0_delphes_events.lhco"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00029139209100000003, '[pb]')


In [42]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(0,10):

    file_path = Folder + "output_ggHH_BP279_42_{:1d}_delphes_events.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13073                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP279/output_ggHH_BP279_42_0_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13073/13073 [00:04<00:00, 2915.15it/s]


('run: ', 0, '   # events ok so far: ', 1002)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13040                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP279/output_ggHH_BP279_42_1_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13040/13040 [00:03<00:00, 3380.22it/s]


('run: ', 1, '   # events ok so far: ', 1992)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13085                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP279/output_ggHH_BP279_42_2_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


...

('run: ', 7, '   # events ok so far: ', 8051)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13180                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP279/output_ggHH_BP279_42_8_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13180/13180 [00:04<00:00, 3042.46it/s]


('run: ', 8, '   # events ok so far: ', 9090)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13005                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP279/output_ggHH_BP279_42_9_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13005/13005 [00:03<00:00, 3506.34it/s]

('run: ', 9, '   # events ok so far: ', 10103)
 
('Total initial events: ', 130749)
('Total events after cuts: ', 10103)


In [43]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 130749)
('Total events after cuts: ', 10103)


In [44]:
initial_evs = 130749
events_ok = 10103

In [45]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.02251592207491453, '[fb]')
('Events expected: ', 67.5477662247436, '    for L=', 3000, ' [fb-1]')


## BP315 (done)

In [46]:
BPnum = 315

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP' + str(BPnum) + '/'
file_path = Folder + "output_ggHH_BP315_38_0_delphes_events.lhco"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.00308735994, '[pb]')


In [47]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(0,10):

    file_path = Folder + "output_ggHH_BP315_38_{:1d}_delphes_events.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13259                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP315/output_ggHH_BP315_38_0_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13259/13259 [00:05<00:00, 2475.53it/s]


('run: ', 0, '   # events ok so far: ', 1282)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13296                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP315/output_ggHH_BP315_38_1_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13296/13296 [00:05<00:00, 2545.32it/s]


('run: ', 1, '   # events ok so far: ', 2602)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13316                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP315/output_ggHH_BP315_38_2_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


...

('run: ', 7, '   # events ok so far: ', 10416)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13296                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP315/output_ggHH_BP315_38_8_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13296/13296 [00:04<00:00, 2828.45it/s]


('run: ', 8, '   # events ok so far: ', 11716)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13355                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP315/output_ggHH_BP315_38_9_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13355/13355 [00:04<00:00, 3114.96it/s]

('run: ', 9, '   # events ok so far: ', 13003)
 
('Total initial events: ', 133137)
('Total events after cuts: ', 13003)


In [48]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 133137)
('Total events after cuts: ', 13003)


In [49]:
initial_evs = 133137
events_ok = 13003

In [50]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.30153106424074455, '[fb]')
('Events expected: ', 904.5931927222337, '    for L=', 3000, ' [fb-1]')


## BP455 (done)

In [51]:
BPnum = 455

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP' + str(BPnum) + '/'
file_path = Folder + "output_ggHH_BP455_41_0_delphes_events.lhco"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000297734276, '[pb]')


In [52]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(0,11):

    file_path = Folder + "output_ggHH_BP455_41_{:1d}_delphes_events.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13102                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP455/output_ggHH_BP455_41_0_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13102/13102 [00:03<00:00, 3411.83it/s]


('run: ', 0, '   # events ok so far: ', 1013)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13019                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP455/output_ggHH_BP455_41_1_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13019/13019 [00:03<00:00, 3537.05it/s]


('run: ', 1, '   # events ok so far: ', 2007)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13044                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP455/output_ggHH_BP455_41_2_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


...

('run: ', 8, '   # events ok so far: ', 9152)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13070                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP455/output_ggHH_BP455_41_9_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13070/13070 [00:03<00:00, 3604.88it/s]


('run: ', 9, '   # events ok so far: ', 10133)
+------------------+-------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13050                                                                                                                   |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP455/output_ggHH_BP455_41_10_delphes_events.lhco |
+------------------+-------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13050/13050 [00:03<00:00, 3534.61it/s]

('run: ', 10, '   # events ok so far: ', 11129)
 
('Total initial events: ', 143744)
('Total events after cuts: ', 11129)


In [53]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 143744)
('Total events after cuts: ', 11129)


In [54]:
initial_evs = 143744
events_ok = 11129

In [55]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.023051290889386688, '[fb]')
('Events expected: ', 69.15387266816006, '    for L=', 3000, ' [fb-1]')


## BP502 (not done)

In [56]:
BPnum = 502

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP' + str(BPnum) + '/'
file_path = Folder + "output_ggHH_BP502_44_0_delphes_events.lhco"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000397166296, '[pb]')


In [57]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(0,10):

    file_path = Folder + "output_ggHH_BP502_44_{:1d}_delphes_events.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13114                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP502/output_ggHH_BP502_44_0_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13114/13114 [00:04<00:00, 2822.38it/s]


('run: ', 0, '   # events ok so far: ', 1116)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13105                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP502/output_ggHH_BP502_44_1_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13105/13105 [00:04<00:00, 3006.75it/s]


('run: ', 1, '   # events ok so far: ', 2266)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13071                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP502/output_ggHH_BP502_44_2_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


...

('run: ', 7, '   # events ok so far: ', 8856)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13171                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP502/output_ggHH_BP502_44_8_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13171/13171 [00:03<00:00, 3518.34it/s]


('run: ', 8, '   # events ok so far: ', 9955)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13098                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP502/output_ggHH_BP502_44_9_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13098/13098 [00:04<00:00, 3198.97it/s]

('run: ', 9, '   # events ok so far: ', 11088)
 
('Total initial events: ', 131407)
('Total events after cuts: ', 11088)


In [58]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 131407)
('Total events after cuts: ', 11088)


In [59]:
initial_evs = 131407
events_ok = 11088

In [60]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.033512521327235235, '[fb]')
('Events expected: ', 100.5375639817057, '    for L=', 3000, ' [fb-1]')


## BP563 (not done)

In [61]:
BPnum = 563

Folder = '/home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP' + str(BPnum) + '/'
file_path = Folder + "output_ggHH_BP563_43_0_delphes_events.lhco"

MG_cross_BP = get_cross_section_from_lhco(file_path)

print("Cross section (MG) = ", MG_cross_BP, "[pb]")

('Cross section (MG) = ', 0.000425348194, '[pb]')


In [62]:

# File to save the processed data:
FileSave = h5py.File("../DATA/signal/BP" + str(BPnum) + ".h5", "a")



# prepare what you  want to save
ensure_vec3("photon")
ensure_vec3("btag")
ensure_vec3("jet")
ensure_vec2("MET")

# number of things already saved in FileSave (to avoid overwritting data already saved)
N_saved = FileSave["photon_pt"].shape[0]



# event counter for this file
initial_evs = 0
events_ok = 0


for i_run in range(0,11):

    file_path = Folder + "output_ggHH_BP563_43_{:1d}_delphes_events.lhco".format(i_run)

    # If the file does not exist, skip and go to the next one
    if not os.path.isfile(file_path):
        print("[WARNING] File not found: {} → skipping".format(file_path))
        continue
        
    datarun = LHCO_reader.Events(f_name=file_path)
    
    # show file characteristics and number of generated events
    print(datarun)
    initial_evs += len(datarun)


    for i_ev, event in tqdm(enumerate(datarun), total=len(datarun), desc="Processing events"): # for the progress bar

        #####################
        # BASIC PARTICLE ID #
        #####################

        index_photon, index_e, index_mu, index_tau, index_jet, index_btag = basic_id_cuts(event)


        
        ############################
        # ESPECIFIC SELECTION CUTS #
        ############################

        passed, index_photon_cut, index_e_cut, index_mu_cut, index_tau_cut, index_jet_cut, index_btag_cut = selection_cuts(event, index_photon, index_e, index_mu, index_tau, index_jet, index_btag)

        if not passed:
            continue # skip and go to the next event
            


        ##############
        # SAVE EVENT #
        ##############

        photon_pts  = [event["photon"][j]["PT"]  for j in index_photon_cut]
        photon_etas = [event["photon"][j]["eta"] for j in index_photon_cut]
        photon_phis = [event["photon"][j]["phi"] for j in index_photon_cut]
        
        btag_pts  = [event["jet"][j]["PT"]  for j in index_btag_cut]
        btag_etas = [event["jet"][j]["eta"] for j in index_btag_cut]
        btag_phis = [event["jet"][j]["phi"] for j in index_btag_cut]

        jet_pts  = [event["jet"][j]["PT"]  for j in index_jet_cut]
        jet_etas = [event["jet"][j]["eta"] for j in index_jet_cut]
        jet_phis = [event["jet"][j]["phi"] for j in index_jet_cut]

        MET_pts  = [event["MET"][0]["PT"]]
        MET_phis = [event["MET"][0]["phi"]]

        # save the event
        N_saved = save_event(   FileSave, N_saved,
                                photon_pts, photon_etas, photon_phis,
                                btag_pts, btag_etas, btag_phis,
                                jet_pts, jet_etas, jet_phis,
                                MET_pts, MET_phis                       )



        # count the number of events that passed everything so far
        events_ok += 1


    pass # for the progress bar
        

    print('run: ',i_run, '   # events ok so far: ' , events_ok)

# don't  forget to close the .h5
FileSave.close()

print(' ')
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13209                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP563/output_ggHH_BP563_43_0_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13209/13209 [00:04<00:00, 3283.93it/s]


('run: ', 0, '   # events ok so far: ', 1121)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13193                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP563/output_ggHH_BP563_43_1_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13193/13193 [00:04<00:00, 3161.57it/s]


('run: ', 1, '   # events ok so far: ', 2249)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13181                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP563/output_ggHH_BP563_43_2_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


...

('run: ', 8, '   # events ok so far: ', 9952)
+------------------+------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13083                                                                                                                  |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP563/output_ggHH_BP563_43_9_delphes_events.lhco |
+------------------+------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13083/13083 [00:03<00:00, 3334.11it/s]


('run: ', 9, '   # events ok so far: ', 11061)
+------------------+-------------------------------------------------------------------------------------------------------------------------+
| Number of events | 13167                                                                                                                   |
| Description      | /home/andres/CompuTools/Programas/MG5_aMC_v3_5_11/DATACluster/LaPlata/BP563/output_ggHH_BP563_43_10_delphes_events.lhco |
+------------------+-------------------------------------------------------------------------------------------------------------------------+


Processing events: 100%|██████████| 13167/13167 [00:04<00:00, 3274.20it/s]

('run: ', 10, '   # events ok so far: ', 12164)
 
('Total initial events: ', 144642)
('Total events after cuts: ', 12164)


In [63]:
print('Total initial events: ', initial_evs)
print('Total events after cuts: ', events_ok)

('Total initial events: ', 144642)
('Total events after cuts: ', 12164)


In [64]:
initial_evs = 144642
events_ok = 12164

In [65]:
cross_fb = MG_cross_BP*1000
aceptancia = 1.*events_ok/initial_evs
luminosity= 3000

fid_cross = cross_fb * aceptancia
Tot_ev_expected = cross_fb * aceptancia * luminosity

print('Fiducial cross section: ', fid_cross, "[fb]")
print('Events expected: ', Tot_ev_expected, "    for L=", luminosity, " [fb-1]")

('Fiducial cross section: ', 0.03577062977431175, '[fb]')
('Events expected: ', 107.31188932293526, '    for L=', 3000, ' [fb-1]')
